# Lecture 1 — Introduction to Deep Learning for Computer Vision

**Course:** UMich EECS 498-007 / 598-005, *Deep Learning for Computer Vision* (Justin Johnson), Fall 2019 (the public YouTube playlist).  
**Merged with:** Stanford CS231n 2026, Lecture 1: [part 1 (history)](https://cs231n.stanford.edu/slides/2026/lecture_1_part_1.pdf), [part 2 (overview)](https://cs231n.stanford.edu/slides/2026/lecture_1_part_2.pdf).

> **About the sources.** The UMich slides couldn't be read directly, so UMich-specific content is based on knowledge of Justin Johnson's lecture. The CS231n 2026 slides were read in full; most of their history slides say *"Slide inspiration: Justin Johnson"*, so both lectures share the same backbone. **[CS231n 2026]** marks content that appears only there; **🔎** marks material beyond both lectures.

### How to read this notebook
This lecture is a **history of computer vision**. Instead of only reading about each historical idea, you'll **build** it, in small steps:

* **Concept cells** (text) explain an idea and the math behind it.
* **Code cells** are short, and do **one thing each**. Every function has a comment that says what it does, what goes in, and what comes out.
* Before an algorithm runs on a real image, it runs on a **tiny example you could check by hand**, with the intermediate numbers printed.
* **"What to notice"** notes after a result tell you what the output means.
* **✍️ Try it** boxes suggest small changes to make and re-run.

Run the cells **in order**, top to bottom: later cells use variables from earlier ones.

### Contents
| § | Topic | What you'll build |
|---|---|---|
| 1 | What is computer vision? | An image, as a computer stores it |
| 2 | Why vision? Eyes and cameras | 🔎 A pinhole camera that renders a 3-D cube |
| 3 | A history of computer vision | Simple & complex cells; Roberts' edges; Marr's 2½-D sketch; parts models; **Canny** edges; **normalized cuts**; **SIFT** matching; **Viola–Jones** face detection |
| 4 | Datasets and ImageNet | The ImageNet error-rate chart |
| 5 | A history of deep learning | The **perceptron** and its limits; backprop solving XOR; **AlexNet's** size and training cost |
| 6 | Deep learning everywhere; the three ingredients | — |
| 7 | Limits, harms, and benefits | Evaluating fairness across groups |
| 8 | Course overview | — |
| 9 | 🔎 Practical setup: images as tensors | NumPy / PyTorch conventions |

### Setup
We use four libraries throughout:
* **NumPy** (`np`): arrays of numbers and fast math on them. Every image in this notebook is a NumPy array.
* **Matplotlib** (`plt`): plotting and showing images.
* **SciPy**: ready-made image filters (`ndimage`) and linear algebra (`eigh`, an eigenvalue solver).
* **Matplotlib's `Path`**: tests whether points lie inside a polygon (used to "paint" the faces of a 3-D cube).

In [ ]:
import numpy as np                      # arrays and math
import matplotlib.pyplot as plt         # plotting
from matplotlib.path import Path        # point-in-polygon tests (for drawing filled shapes)
from scipy import ndimage               # image filters: blur, gradients, rotation, labeling
from scipy.linalg import eigh           # eigenvalue solver (used by normalized cuts)
import math, time

np.set_printoptions(precision=3, suppress=True)   # print arrays with 3 decimals, no scientific notation

---
## 1. What is computer vision?

> **Computer vision** is building artificial systems that **process, perceive, and reason about visual data**.

**Where it fits:**
* **Artificial intelligence (AI):** the broad goal of building intelligent machines.
* **Machine learning (ML):** the part of AI where machines **learn from data** instead of following hand-written rules.
* **Deep learning (DL):** the part of ML that uses **neural networks with many layers** ("deep" = many layers).
* **Computer vision (CV):** overlaps all three. **This course sits where computer vision and deep learning intersect.**

Computer vision also borrows from **computer science**, **mathematics** (linear algebra, calculus, probability), **physics** (how light and lenses form images), **biology and neuroscience** (how animal vision works), and **psychology** (how people perceive).

**Computer vision is everywhere:** face unlock and photo search on phones, driver assistance and self-driving cars, medical scans, satellite and telescope imagery, factory inspection, robots, augmented reality, and, since the 2020s, image and video *generation*.

### 1.1 What a computer actually "sees"
To a computer, a picture is **a grid of numbers**:
* A **grayscale** image of height $H$ and width $W$ is an $H\times W$ array: one brightness number per pixel.
* A **color** image is $H\times W\times3$: three numbers per pixel, for **red, green, and blue** (RGB).
* The numbers are usually whole numbers from **0 (dark) to 255 (bright)**, stored as the type `uint8` ("unsigned 8-bit integer": exactly the range 0–255).

Let's build a tiny 6×8 color image by hand. **Step 1:** create an all-black image.

In [ ]:
H, W = 6, 8                                     # 6 rows (height) and 8 columns (width)
img = np.zeros((H, W, 3), dtype=np.uint8)       # 3 channels (R, G, B), all zeros = black

print("shape:", img.shape)                      # (height, width, channels)
print("data type:", img.dtype)                  # uint8: whole numbers 0..255
print("pixel at row 0, column 0 (R, G, B):", img[0, 0])

**Step 2: paint regions.** NumPy indexing `img[rows, columns] = color` sets every pixel in a block at once. `:` means "all", and `a:b` means "from index a up to (not including) b".

In [ ]:
img[:, :W // 2] = [220, 40, 40]       # all rows, left half of the columns  -> red
img[:, W // 2:] = [40, 40, 220]       # all rows, right half of the columns -> blue
img[2:4, 3:5] = [255, 255, 255]       # rows 2-3, columns 3-4 -> a white 2x2 square

print("top-left pixel    :", img[0, 0])     # red
print("top-right pixel   :", img[0, 7])     # blue
print("pixel in the square:", img[2, 3])    # white

**Step 3: look at the numbers the computer stores**, one channel at a time, and at the picture we see.

In [ ]:
print("RED channel (img[:, :, 0]):")
print(img[:, :, 0])

fig, ax = plt.subplots(1, 4, figsize=(12, 2.5))
ax[0].imshow(img); ax[0].set_title("what we see")
for c, name in enumerate(["R", "G", "B"]):
    ax[c + 1].imshow(img[:, :, c], cmap="gray", vmin=0, vmax=255)   # show one channel as brightness
    ax[c + 1].set_title(f"{name} channel")
for a in ax: a.axis("off")
plt.show()

**What to notice:** the red channel is 220 on the left (red region), 40 on the right, and 255 in the white square (white = all three channels at maximum). We see "a white square on a red-and-blue background"; the computer sees 144 numbers. That difference between the numbers and their meaning is called the **semantic gap** (Lecture 2), and closing it is what this whole course is about.

🔎 **A common bug: `uint8` overflow.** `uint8` can't go above 255 or below 0, so arithmetic **wraps around** instead of stopping. Convert to a float type before doing math on images:

In [ ]:
pixel = np.array([250], dtype=np.uint8)
print("uint8: 250 + 10 =", pixel + np.uint8(10), "   <- wraps around to 4!")
print("float: 250 + 10 =", pixel.astype(np.float32) + 10, " <- correct; clip back to 0..255 when saving")

---
## 2. Why vision? Eyes and cameras

**Evolution's "Big Bang": the Cambrian explosion, ~540 million years ago.** In a short geological period, animal life diversified dramatically. One hypothesis (Andrew Parker's "light switch" theory) is that the **first eyes** triggered it: once animals could see, hunting and hiding became an evolutionary arms race. Today vision is the main sense of most intelligent animals. **[CS231n 2026]**: human development depends on visual intelligence, and humans build civilization using it.

**The camera.** A **camera obscura** ("dark room") is a dark box with a tiny hole: light from the scene passes through the hole and projects an upside-down image on the opposite wall. Leonardo da Vinci described it in the 1500s, and Gemma Frisius drew one in 1545. Every camera since works this way, with a lens replacing the hole.

### 2.1 🔎 The pinhole camera model
The math of the camera obscura is the starting point of 3-D computer vision. Put the pinhole at the origin, looking along the $Z$ axis. A point at position $(X, Y, Z)$ ($Z$ = its distance in front of the camera) lands on the image at
$$x = f\,\frac{X}{Z} + c_x,\qquad y = f\,\frac{Y}{Z} + c_y$$
* $f$ is the **focal length** (in pixels): how strongly the camera zooms.
* $(c_x, c_y)$ is the **image center**, so the camera's axis lands in the middle of the picture.
* **Dividing by $Z$ is perspective:** something twice as far away looks half as big.

**Where the formula comes from (similar triangles):** a ray from the point through the pinhole hits the image plane at distance $f$. The triangle (pinhole, image point) is a scaled copy of the triangle (pinhole, scene point), so $x/f = X/Z$.

**Step 1: project one point**, then check that doubling the distance halves the offset from the center.

In [ ]:
def project(point_3d, f=150.0, center=(64.0, 64.0)):
    # Project a 3-D point (X, Y, Z) in camera coordinates to 2-D pixel coordinates (x, y)
    # using the pinhole model x = f * X / Z + cx,  y = f * Y / Z + cy.
    X, Y, Z = point_3d
    return np.array([f * X / Z + center[0], f * Y / Z + center[1]])

near = project((1.0, 0.5, 2.0))       # a point 2 units in front of the camera
far = project((1.0, 0.5, 4.0))        # the same X, Y, but twice as far away
print("near point lands at", near, "-> offset from center", near - 64)
print("far  point lands at", far, "-> offset from center", far - 64, "  (half as big: perspective)")

**Step 2: describe a cube.** A cube centered at the origin has 8 corners, all combinations of $\pm1$ in $X$, $Y$, $Z$, and 6 square faces, each listed by its 4 corner indices.

In [ ]:
corners = np.array([[x, y, z] for x in (-1, 1) for y in (-1, 1) for z in (-1, 1)], float)   # 8 corners
faces = [[0, 1, 3, 2], [4, 5, 7, 6],      # x = -1 face, x = +1 face
         [0, 1, 5, 4], [2, 3, 7, 6],      # y = -1 face, y = +1 face
         [0, 2, 6, 4], [1, 3, 7, 5]]      # z = -1 face, z = +1 face
for i, c in enumerate(corners):
    print(f"corner {i}: {c}")

**Step 3: turn the cube and place it in front of the camera.** A **rotation matrix** spins points around an axis without stretching them. Rotating by angle $\theta$ around the $Y$ (vertical) axis, and by $\phi$ around the $X$ axis:
$$R_y = \begin{bmatrix}\cos\theta & 0 & \sin\theta\\ 0 & 1 & 0\\ -\sin\theta & 0 & \cos\theta\end{bmatrix},\qquad R_x = \begin{bmatrix}1 & 0 & 0\\ 0 & \cos\phi & -\sin\phi\\ 0 & \sin\phi & \cos\phi\end{bmatrix}$$
Then we push the cube 4 units forward (add 4 to $Z$) so the whole cube is in front of the pinhole.

In [ ]:
def rotation(angle_y, angle_x):
    # Rotation matrix that turns points by angle_y around the vertical axis, then angle_x around the horizontal axis.
    cy, sy, cx, sx = np.cos(angle_y), np.sin(angle_y), np.cos(angle_x), np.sin(angle_x)
    Ry = np.array([[cy, 0, sy], [0, 1, 0], [-sy, 0, cy]])
    Rx = np.array([[1, 0, 0], [0, cx, -sx], [0, sx, cx]])
    return Rx @ Ry

R = rotation(0.6, 0.45)
cube_cam = corners @ R.T + np.array([0, 0, 4.0])       # rotate every corner, then move 4 units in front of the camera
print("rotation keeps lengths (R is orthonormal):", np.allclose(R @ R.T, np.eye(3)))
print("all corners in front of the camera (Z > 0):", bool(np.all(cube_cam[:, 2] > 0)))

**Step 4: project all corners and draw the edges** (a "wireframe"). This is already a perspective image of a cube.

In [ ]:
corners_2d = np.array([project(p) for p in cube_cam])
plt.figure(figsize=(3.5, 3.5))
for face in faces:
    loop = corners_2d[face + [face[0]]]          # the face's 4 corners, back to the first to close the square
    plt.plot(loop[:, 0], loop[:, 1], "k-")
plt.gca().invert_yaxis(); plt.gca().set_aspect("equal"); plt.title("wireframe cube (pinhole projection)"); plt.show()

**Step 5: render solid faces.** To draw a solid cube we need two more ideas:
* **Which faces are visible?** A face is visible if its outward-facing direction (its **normal** vector) points toward the camera. We get a normal from the **cross product** of two of the face's edges, flip it if it points into the cube, and check its dot product with the direction to the camera.
* **Painter's algorithm:** paint faces from far to near, so nearer faces cover farther ones.

We also record a **label map**: which face each pixel shows. The borders between labels are the cube's **true edges**, which we'll use later to grade edge detectors.

In [ ]:
def render_cube(size=128, f=150.0, angle_y=0.6, angle_x=0.45, dist=4.0, shades=(0.85, 0.55, 0.3), bg=0.1):
    # Render a shaded cube with the pinhole model.
    # Returns: image (size x size, brightness 0..1) and labels (0 = background, 1..3 = which visible face).
    P = corners @ rotation(angle_y, angle_x).T + np.array([0, 0, dist])        # cube corners in camera coordinates
    xy = np.array([project(p, f, (size / 2, size / 2)) for p in P])           # their pixel positions
    img = np.full((size, size), bg); labels = np.zeros((size, size), int)
    yy, xx = np.mgrid[0:size, 0:size]
    pixel_centers = np.c_[xx.ravel() + 0.5, yy.ravel() + 0.5]

    visible = []
    cube_center = P.mean(0)
    for fc in faces:
        face_center = P[fc].mean(0)
        normal = np.cross(P[fc[1]] - P[fc[0]], P[fc[2]] - P[fc[0]])            # perpendicular to the face
        if normal @ (face_center - cube_center) < 0:                            # make it point OUT of the cube
            normal = -normal
        if normal @ (-face_center) > 0:                                         # does it point toward the camera (origin)?
            visible.append((face_center[2], fc))                                # remember its depth for sorting

    for k, (_, fc) in enumerate(sorted(visible, reverse=True)):                 # far faces first (painter's algorithm)
        inside = Path(xy[fc]).contains_points(pixel_centers).reshape(size, size)
        img[inside] = shades[k % len(shades)]
        labels[inside] = k + 1
    return img, labels

cube, cube_labels = render_cube()
print("visible faces:", cube_labels.max())

**Step 6: find the true edges** (pixels where the label changes from one pixel to the next) and show the result.

In [ ]:
def label_boundaries(labels):
    # True where a pixel's label differs from the pixel above it or to its left.
    down = np.abs(np.diff(labels, axis=0, prepend=labels[:1])) > 0      # compare with the row above
    right = np.abs(np.diff(labels, axis=1, prepend=labels[:, :1])) > 0  # compare with the column to the left
    return down | right

true_edges = label_boundaries(cube_labels)
fig, ax = plt.subplots(1, 2, figsize=(7, 3.5))
ax[0].imshow(cube, cmap="gray", vmin=0, vmax=1); ax[0].set_title("rendered cube")
ax[1].imshow(true_edges, cmap="gray"); ax[1].set_title(f"true edges ({true_edges.sum()} pixels)")
for a in ax: a.axis("off")
plt.show()

**✍️ Try it:** change `angle_y` or `f` in `render_cube` and re-run. Larger `f` zooms in; small `dist` exaggerates perspective.

---
## 3. A brief history of computer vision

The milestones, as both lectures present them:

| Year | Milestone | The idea |
|---|---|---|
| 1959 | **Hubel & Wiesel** | The visual cortex has "simple" and "complex" cells |
| 1963 | **Larry Roberts** | Find edges, then reason about 3-D blocks |
| 1966 | **MIT Summer Vision Project** | "Solve vision in one summer" |
| 1970s | **David Marr** | Vision as a series of representations |
| 1970s | **Recognition via parts** | Objects = parts + how they connect |
| 1980s | **Recognition via edges** | Canny (1986), Lowe (1987) |
| ~1987–93 | **AI winter** | Funding and enthusiasm collapse |
| 1990s | **Recognition via grouping** | Normalized cuts (1997) |
| 2000s | **Recognition via matching** | SIFT (1999) |
| 2001 | **Viola & Jones** | Real-time face detection with machine learning |
| 2004–09 | **Datasets** | Caltech101, PASCAL VOC, ImageNet |

### 3.1 Hubel and Wiesel (1959): how the visual cortex works
**The experiment:** Hubel and Wiesel recorded the electrical activity of single neurons in a cat's visual cortex while showing the cat simple patterns, such as bars of light at different angles (Nobel Prize, 1981).

**What they found:**
* **Simple cells** fire for a bar at **one particular orientation**, in **one particular position**.
* **Complex cells** fire for the right orientation **over a range of positions**: they're tolerant to small shifts (**translation invariance**).
* The visual system is a **hierarchy**: simple features first, combined into more complex ones.

**Why it matters for this course:** "simple cells detect oriented edges at a location" → **convolution filters**; "complex cells combine simple cells over nearby locations" → **pooling**. That is exactly the structure of a convolutional neural network (Lecture 7).

**Modeling a simple cell.** A good mathematical model of a simple cell's preferences is a **Gabor filter**: a striped (sinusoidal) pattern inside a soft round window (a Gaussian). The cell's response to an image is the **dot product** of the image patch with the filter: big when the patch looks like the filter.
$$g_\theta(x, y) = \underbrace{\exp\left(-\frac{x'^2 + y'^2}{2\sigma^2}\right)}_{\text{round window}}\ \underbrace{\cos\left(\frac{2\pi x'}{\lambda}\right)}_{\text{stripes}},\qquad x' = x\cos\theta + y\sin\theta,\ \ y' = -x\sin\theta + y\cos\theta$$
* $\theta$: the preferred **orientation**. ($x', y'$ are the coordinates rotated by $\theta$, so the stripes turn with it.)
* $\sigma$: the window's **size**; $\lambda$: the stripes' **spacing**.

**Step 1: build the filter**, and look at it at four orientations.

In [ ]:
def gabor(theta, size=15, sigma=3.0, wavelength=8.0):
    # A (size x size) Gabor filter: stripes at orientation theta, inside a Gaussian window.
    # We subtract its mean so a blank (uniform) patch gives zero response.
    half = size // 2
    y, x = np.mgrid[-half:half + 1, -half:half + 1].astype(float)   # coordinates centered at 0
    x_rot = x * np.cos(theta) + y * np.sin(theta)                    # rotate the coordinates by theta
    y_rot = -x * np.sin(theta) + y * np.cos(theta)
    window = np.exp(-(x_rot ** 2 + y_rot ** 2) / (2 * sigma ** 2))    # soft round window
    stripes = np.cos(2 * np.pi * x_rot / wavelength)                  # sinusoidal stripes
    g = window * stripes
    return g - g.mean()

fig, ax = plt.subplots(1, 4, figsize=(8, 2.2))
for a, deg in zip(ax, [0, 45, 90, 135]):
    a.imshow(gabor(np.radians(deg)), cmap="RdBu"); a.set_title(f"theta = {deg} deg"); a.axis("off")
plt.suptitle("Gabor filters: red = positive weights (likes light), blue = negative (likes dark)", fontsize=9); plt.show()

**Step 2: make test stimuli**, bright bars at a chosen angle, like the bars of light in the experiment. The `offset` slides the bar sideways (perpendicular to its length).

In [ ]:
def bar_stimulus(theta, offset=0.0, size=31, width=2.0):
    # A (size x size) image: 1 inside a bright bar through the center at angle theta, 0 elsewhere.
    # offset moves the bar sideways by that many pixels.
    half = size // 2
    y, x = np.mgrid[-half:half + 1, -half:half + 1].astype(float)
    distance_from_bar = x * np.cos(theta) + y * np.sin(theta) - offset   # signed distance across the bar
    return (np.abs(distance_from_bar) < width).astype(float)

fig, ax = plt.subplots(1, 3, figsize=(6, 2.2))
for a, (deg, off) in zip(ax, [(0, 0), (45, 0), (0, 4)]):
    a.imshow(bar_stimulus(np.radians(deg), off), cmap="gray"); a.set_title(f"angle {deg}, offset {off}", fontsize=8); a.axis("off")
plt.show()

**Step 3: a simple cell = one filter at one location.** Its response is the dot product of the filter with the image patch at the center, passed through $\max(0, \cdot)$ (a neuron's firing rate can't be negative). Notice the operation: **multiply element by element, then sum**, the same operation a convolution layer performs at every position.

In [ ]:
def simple_cell(stimulus, theta):
    # Response of a simple cell that prefers orientation theta and looks only at the CENTER of the image.
    k = gabor(theta)
    c, h = stimulus.shape[0] // 2, k.shape[0] // 2
    patch = stimulus[c - h:c + h + 1, c - h:c + h + 1]        # the region the cell looks at
    return max(0.0, float(np.sum(patch * k)))                  # dot product, then no negative firing

preferred = 0.0                                                # this cell prefers vertical bars
for deg in [0, 20, 45, 90]:
    print(f"bar at {deg:2d} degrees -> simple cell response {simple_cell(bar_stimulus(np.radians(deg)), preferred):6.2f}")

**Step 4: a complex cell = the maximum of many simple cells at nearby positions.** We compute the simple-cell response at *every* position with `ndimage.correlate` (it slides the filter across the image, exactly like a convolution layer), and take the largest response within a small neighborhood. Taking the max over positions is **max pooling**.

In [ ]:
def complex_cell(stimulus, theta, pool=6):
    # Max of rectified simple-cell responses over a (2*pool+1) x (2*pool+1) neighborhood of the center.
    responses = ndimage.correlate(stimulus, gabor(theta), mode="constant")   # simple cells at every position
    responses = np.maximum(responses, 0)
    c = stimulus.shape[0] // 2
    return float(responses[c - pool:c + pool + 1, c - pool:c + pool + 1].max())

for shift in [0, 2, 4]:
    s = bar_stimulus(preferred, offset=shift)
    print(f"vertical bar shifted by {shift} px: simple cell {simple_cell(s, preferred):5.2f}   complex cell {complex_cell(s, preferred):5.2f}")

**Step 5: the full picture**: orientation tuning (how response depends on the bar's angle) and position tolerance (how it depends on the bar's shift).

In [ ]:
angles = np.linspace(-np.pi / 2, np.pi / 2, 37)
shifts = np.arange(-6, 7)
tuning = [simple_cell(bar_stimulus(a), preferred) for a in angles]
simple_vs_shift = [simple_cell(bar_stimulus(preferred, s), preferred) for s in shifts]
complex_vs_shift = [complex_cell(bar_stimulus(preferred, s), preferred) for s in shifts]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(np.degrees(angles), tuning, "o-", ms=3)
ax[0].set_xlabel("bar orientation (degrees)"); ax[0].set_ylabel("response"); ax[0].set_title("simple cell: orientation tuning")
ax[1].plot(shifts, simple_vs_shift, "o-", label="simple cell (one position)")
ax[1].plot(shifts, complex_vs_shift, "s-", label="complex cell (max over positions)")
ax[1].set_xlabel("bar shift (pixels)"); ax[1].set_title("position tolerance"); ax[1].legend()
plt.tight_layout(); plt.show()

**What to notice:**
* The simple cell responds strongly only near its preferred orientation (0°) and is silent at 90°: it's an **oriented edge/bar detector**.
* Shift the bar by 4 pixels and the simple cell goes **silent**, while the complex cell **keeps responding**: taking the max over nearby positions buys **translation invariance**.

Keep this two-step pattern, *detect local features everywhere, then pool over nearby positions*, in mind for Lecture 7.

### 3.2 Larry Roberts (1963): the blocks world
Roberts' MIT PhD thesis, *Machine Perception of Three-Dimensional Solids*, is often called the first computer-vision thesis. Given a photo of simple blocks, it (a) found **edges**, (b) picked out **corner points**, and (c) inferred the 3-D shapes.

**What is an edge?** A place where brightness **changes quickly**. Change is measured by **differences between neighboring pixels** (a discrete derivative). Roberts used two tiny 2×2 filters, now called the **Roberts cross**, which measure the change along the two diagonals:
$$G_1 = \begin{bmatrix}1 & 0\\0 & -1\end{bmatrix}\ (\text{top-left} - \text{bottom-right}),\qquad G_2 = \begin{bmatrix}0 & 1\\-1 & 0\end{bmatrix}\ (\text{top-right} - \text{bottom-left})$$
The **edge strength** at a pixel is $\sqrt{g_1^2 + g_2^2}$: large if brightness changes in *either* diagonal direction.

**Worked example first:** a 4×4 image with a sharp vertical boundary between dark (0) and bright (1).

In [ ]:
def roberts_cross(I):
    # Edge strength using the Roberts cross operator. Output has the same size as I (last row/column padded with 0).
    g1 = I[:-1, :-1] - I[1:, 1:]             # top-left minus bottom-right of each 2x2 block
    g2 = I[:-1, 1:] - I[1:, :-1]             # top-right minus bottom-left
    strength = np.sqrt(g1 ** 2 + g2 ** 2)
    return np.pad(strength, ((0, 1), (0, 1)))

tiny = np.array([[0, 0, 1, 1]] * 4, float)
print("tiny image:\n", tiny)
print("Roberts edge strength:\n", roberts_cross(tiny))

The strength is about 1.41 exactly where the dark-to-bright boundary is (both diagonals see a difference of 1: $\sqrt{1^2 + 1^2}$), and 0 elsewhere. ✔

**To grade an edge detector** we compare its edge pixels with the true edges from our renderer, using two standard measures (with a 1-pixel tolerance, since an edge one pixel off is still basically right):
* **Precision:** of the pixels the detector marked as edges, what fraction are real edges? (Low precision = many false alarms.)
* **Recall:** of the real edge pixels, what fraction did the detector find? (Low recall = missed edges.)
* **F1 score:** one number combining them, $F_1 = \frac{2\cdot\text{precision}\cdot\text{recall}}{\text{precision} + \text{recall}}$. 1.0 is perfect.

In [ ]:
def edge_f1(predicted, truth, tolerance=1):
    # Precision, recall, and F1 of predicted edge pixels vs. true edge pixels, allowing `tolerance` pixels of slack.
    near_truth = ndimage.binary_dilation(truth, iterations=tolerance)        # true edges, thickened by 1 pixel
    near_pred = ndimage.binary_dilation(predicted, iterations=tolerance)
    precision = (predicted & near_truth).sum() / max(predicted.sum(), 1)    # predicted edges that are near a real one
    recall = (truth & near_pred).sum() / max(truth.sum(), 1)                # real edges that were found
    f1 = 2 * precision * recall / max(precision + recall, 1e-9)
    return f1, precision, recall

Now apply the Roberts cross to the cube, first clean, then with a little camera **noise** (random brightness fluctuations, as every real camera has). We call a pixel an edge if its strength is above a threshold of 0.15.

In [ ]:
noise_rng = np.random.default_rng(0)
noisy_cube = cube + noise_rng.normal(0, 0.08, cube.shape)        # add Gaussian noise with standard deviation 0.08

for name, image in [("clean", cube), ("noisy", noisy_cube)]:
    edges = roberts_cross(image) > 0.15                            # threshold the edge strength
    f1, p, r = edge_f1(edges, true_edges)
    print(f"Roberts on the {name} cube: F1 {f1:.2f}  (precision {p:.2f}, recall {r:.2f})")

fig, ax = plt.subplots(1, 2, figsize=(7, 3.5))
ax[0].imshow(roberts_cross(cube) > 0.15, cmap="gray"); ax[0].set_title("Roberts, clean")
ax[1].imshow(roberts_cross(noisy_cube) > 0.15, cmap="gray"); ax[1].set_title("Roberts, noisy")
for a in ax: a.axis("off")
plt.show()

**What to notice:** on the clean image the detector is nearly perfect, but with a little noise, **precision collapses**: noise makes neighboring pixels differ randomly, and every noisy pixel looks like an edge. Real photos are noisy, textured, and cluttered, which is why the blocks world didn't extend to the real world. §3.6 shows how Canny fixed this.

### 3.3 The MIT Summer Vision Project (1966)
Seymour Papert's MIT memo proposed *"The Summer Vision Project … an attempt to use our summer workers effectively in the construction of a significant part of a visual system."* The plan: in one summer, split images into objects and background and recognize the objects. **More than fifty years later, we're still working on it.** The lesson: vision feels effortless to us, but it is computationally very hard.

### 3.4 David Marr (1970s): stages of visual representation
In his book *Vision* (1982), Marr proposed that seeing builds a sequence of representations:
1. **Input image:** the brightness at each pixel.
2. **Primal sketch:** edges, bars, blobs, ends, boundaries.
3. **2½-D sketch:** for each pixel, **how far** the surface is (depth) and **which way it faces** (surface orientation), as seen *from the viewer's position*.
4. **3-D model:** objects described in their own coordinates, built from parts.

Marr also suggested analyzing any vision system at three **levels**: *what* problem is solved and why (computational), *how* (algorithmic), and *in what hardware* (implementation). It's a useful checklist for any ML system too.

**🔎 Computing a 2½-D sketch.** Suppose we know the depth $Z(x, y)$ at each pixel. The **surface normal** (the direction the surface faces) comes from how fast the depth changes:
$$\mathbf{n}\propto\left(-\frac{\partial Z}{\partial x},\ -\frac{\partial Z}{\partial y},\ 1\right)$$
A flat surface facing the camera has $\mathbf{n} = (0, 0, 1)$; a surface tilting away has a normal leaning sideways. From normals we can compute **shading**: a matte surface's brightness is $\max(0, \mathbf{n}\cdot\mathbf{l})$ for light direction $\mathbf{l}$ (**Lambert's law**: surfaces facing the light are brightest).

**Step 1: a depth map**, a round bump sticking out toward the camera (smaller $Z$ = closer).

In [ ]:
yy, xx = np.mgrid[-1:1:96j, -1:1:96j]                         # a 96 x 96 grid of coordinates from -1 to 1
inside = xx ** 2 + yy ** 2 < 0.6                               # a disk
Z = 2.0 - np.where(inside, np.sqrt(np.clip(0.6 - xx ** 2 - yy ** 2, 0, None)), 0.0)   # hemisphere bump, base at depth 2
print(f"depth at the center: {Z[48, 48]:.2f} (closest),  at the corner: {Z[0, 0]:.2f}")

**Step 2: normals from depth gradients.** `np.gradient` computes $\partial Z/\partial y$ and $\partial Z/\partial x$ by differences between neighbors (the grid spacing is $2/95$). We then normalize each normal to length 1.

In [ ]:
dZ_dy, dZ_dx = np.gradient(Z, 2 / 95)                          # rate of change of depth down and across
normals = np.dstack([-dZ_dx, -dZ_dy, np.ones_like(Z)])         # n = (-dZ/dx, -dZ/dy, 1) at every pixel
normals /= np.linalg.norm(normals, axis=2, keepdims=True)      # unit length
print("normal at the center of the bump:", normals[48, 48].round(3), "(faces the camera)")
print("normal on the bump's left side  :", normals[48, 25].round(3), "(tilted to the side)")

**Step 3: shading from normals**, with light coming from the upper left, and the three representations side by side.

In [ ]:
light = np.array([0.5, -0.5, 0.7]); light /= np.linalg.norm(light)   # unit vector toward the light
shading = np.clip(normals @ light, 0, None)                           # Lambert: brightness = max(0, n . l)

fig, ax = plt.subplots(1, 3, figsize=(11, 3.5))
ax[0].imshow(Z, cmap="viridis"); ax[0].set_title("depth Z (2.5-D)")
ax[1].imshow((normals + 1) / 2); ax[1].set_title("normals (color = direction)")
ax[2].imshow(shading, cmap="gray"); ax[2].set_title("shading from the normals")
for a in ax: a.axis("off")
plt.show()

**What to notice:** the same geometry produces a shaded image that *looks* 3-D. Modern networks run this backward, predicting depth and normals from a single photo (Lecture 17).

### 3.5 Recognition via parts (1970s)
* **Generalized cylinders** (Brooks & Binford, 1979): describe an object as a set of simple 3-D shapes, each made by sweeping a cross-section along an axis (a person = cylinders for the torso, arms, and legs).
* **Pictorial structures** (Fischler & Elschlager, 1973): an object is a set of **parts** connected by **springs**. To find the object in an image, choose part locations $l_1, \dots, l_n$ that minimize
$$E(l_1, \dots, l_n) = \underbrace{\sum_i m_i(l_i)}_{\text{each part should look right}} + \underbrace{\sum_{(i,j)}d_{ij}(l_i, l_j)}_{\text{springs shouldn't stretch}}$$

**Tiny worked example:** a "face" with 3 parts along a line (left eye, nose, right eye), each of which can sit at positions 0–9. Each part has a **match cost** at each position (low = looks right there), and springs prefer the eyes about 3 positions from the nose. We find the best placement by trying all $10^3$ combinations.

In [ ]:
positions = np.arange(10)
match_cost = {                                    # low cost = the image looks like that part at that position
    "left eye":  np.array([5, 5, 1, 4, 5, 5, 2, 5, 5, 5], float),   # two candidate spots: 2 and 6
    "nose":      np.array([5, 5, 5, 5, 5, 1, 5, 5, 5, 5], float),   # clearly at 5
    "right eye": np.array([5, 5, 5, 5, 5, 5, 5, 5, 1, 5], float),   # clearly at 8
}
def spring_cost(a, b, rest=3.0, k=1.0):
    # Quadratic spring: zero when the two parts are `rest` apart, growing as they move away from that distance.
    return k * (abs(a - b) - rest) ** 2

best = None
for le in positions:
    for n in positions:
        for re in positions:
            E = (match_cost["left eye"][le] + match_cost["nose"][n] + match_cost["right eye"][re]
                 + spring_cost(le, n) + spring_cost(n, re))
            if best is None or E < best[0]:
                best = (E, le, n, re)
print(f"best placement: left eye {best[1]}, nose {best[2]}, right eye {best[3]}  (total cost {best[0]:.1f})")
print("the left eye's match cost alone liked positions 2 AND 6; the spring to the nose picks 2 (3 away from the nose)")

**What to notice:** the springs resolve ambiguity: the left-eye detector alone was unsure, but its relationship to the nose decided. (Real systems search 2-D positions efficiently with dynamic programming; this idea came back, with learned part detectors, as Deformable Part Models (Felzenszwalb et al., 2008–2010), the best object detector just before deep learning.) These hand-designed object models were elegant, but didn't work on real photographs.

### 3.6 Recognition via edge detection (1980s): Canny
**John Canny (1986)** designed an edge detector from first principles: it should (1) **find real edges and few false ones**, (2) **locate them accurately**, and (3) give **one response per edge** (thin lines, not thick bands). His algorithm has four steps, which we build one at a time on the noisy cube.

**Step 1: smooth.** Blur with a Gaussian to average away noise before taking differences. `sigma` controls how much: more blur removes more noise but also softens edges.

In [ ]:
smoothed = ndimage.gaussian_filter(noisy_cube, sigma=1.5)
fig, ax = plt.subplots(1, 2, figsize=(7, 3.5))
ax[0].imshow(noisy_cube, cmap="gray"); ax[0].set_title("noisy input")
ax[1].imshow(smoothed, cmap="gray"); ax[1].set_title("after Gaussian smoothing")
for a in ax: a.axis("off")
plt.show()

**Step 2: gradient magnitude and direction.** The **Sobel** filters estimate the horizontal and vertical brightness changes ($g_x$, $g_y$) using 3×3 neighborhoods (sturdier than Roberts' 2×2):
$$S_x = \begin{bmatrix}-1 & 0 & 1\\-2 & 0 & 2\\-1 & 0 & 1\end{bmatrix},\qquad S_y = S_x^\top,\qquad \text{magnitude} = \sqrt{g_x^2 + g_y^2},\qquad \text{direction} = \arctan(g_y/g_x)$$
The **direction** points across the edge (from dark to bright); the edge itself runs perpendicular to it.

In [ ]:
gx = ndimage.sobel(smoothed, axis=1) / 8          # horizontal change (divide by 8 so a 0->1 step gives about 1)
gy = ndimage.sobel(smoothed, axis=0) / 8          # vertical change
magnitude = np.hypot(gx, gy)                      # sqrt(gx^2 + gy^2)
direction = (np.degrees(np.arctan2(gy, gx)) + 180) % 180   # angle in [0, 180): which way brightness changes
print(f"largest gradient magnitude: {magnitude.max():.3f};  median: {np.median(magnitude):.4f}")

fig, ax = plt.subplots(1, 2, figsize=(7, 3.5))
ax[0].imshow(magnitude, cmap="gray"); ax[0].set_title("gradient magnitude")
ax[1].imshow(direction, cmap="hsv"); ax[1].set_title("gradient direction")
for a in ax: a.axis("off")
plt.show()

Edges now show up as **thick bright bands** (the blur spread each edge over several pixels).

**Step 3: non-maximum suppression (NMS)** thins each band to a single pixel: keep a pixel only if its magnitude is the **largest along the gradient direction**, compared with its two neighbors across the edge. We round each direction to one of four (0°, 45°, 90°, 135°) to decide which two neighbors to compare:

| Rounded direction | Compare with the neighbors… |
|---|---|
| 0° (brightness changes left→right) | left and right |
| 45° | the two diagonal neighbors on one diagonal |
| 90° (changes top→bottom) | above and below |
| 135° | the two on the other diagonal |

In [ ]:
def non_max_suppression(magnitude, direction):
    # Keep only pixels whose magnitude is >= both neighbors along the (rounded) gradient direction.
    bins = np.round(direction / 45).astype(int) % 4          # 0: 0 deg, 1: 45, 2: 90, 3: 135
    neighbor_offsets = {0: (0, 1), 1: (1, 1), 2: (1, 0), 3: (1, -1)}   # (row step, column step) for each direction
    padded = np.pad(magnitude, 1)
    H_, W_ = magnitude.shape
    out = np.zeros_like(magnitude)
    for b, (dr, dc) in neighbor_offsets.items():
        ahead = padded[1 + dr:1 + dr + H_, 1 + dc:1 + dc + W_]          # neighbor one step along the direction
        behind = padded[1 - dr:1 - dr + H_, 1 - dc:1 - dc + W_]         # neighbor one step the other way
        keep = (bins == b) & (magnitude >= ahead) & (magnitude >= behind)
        out[keep] = magnitude[keep]
    return out

thin = non_max_suppression(magnitude, direction)
print(f"pixels with magnitude > 0.03: before NMS {np.sum(magnitude > 0.03):,}, after NMS {np.sum(thin > 0.03):,}")

**Step 4: hysteresis thresholding**, with two thresholds:
* Pixels above the **high** threshold are definitely edges ("strong").
* Pixels above the **low** threshold are edges **only if they're connected to a strong pixel** ("weak but supported").

This keeps long, faint edges (supported by a strong part) while dropping isolated noise (weak and alone). `ndimage.label` finds connected groups of weak pixels; we keep the groups that contain at least one strong pixel.

**Tiny 1-D example** first: a row of edge strengths, with low = 0.3 and high = 0.6.

In [ ]:
def hysteresis(strength, low, high):
    # Edges = pixels above `low` that belong to a connected group containing at least one pixel above `high`.
    weak = strength > low
    strong = strength > high
    groups, n_groups = ndimage.label(weak, structure=np.ones((3,) * strength.ndim))   # connected groups of weak pixels
    groups_with_strong = np.unique(groups[strong])
    return weak & np.isin(groups, groups_with_strong)

row = np.array([0.1, 0.4, 0.5, 0.8, 0.4, 0.1, 0.45, 0.4, 0.1])
print("strengths:      ", row)
print("above low (0.3):", (row > 0.3).astype(int))
print("hysteresis:     ", hysteresis(row, 0.3, 0.6).astype(int), " <- the 0.45/0.4 group has no strong pixel, so it's dropped")

**Putting the four steps together**, and comparing with Roberts on the same noisy image:

In [ ]:
def canny(I, sigma=1.5, low=0.03, high=0.06):
    # Canny edge detector: smooth -> gradient -> non-maximum suppression -> hysteresis. Returns a boolean edge map.
    S = ndimage.gaussian_filter(I, sigma)
    gx_, gy_ = ndimage.sobel(S, axis=1) / 8, ndimage.sobel(S, axis=0) / 8
    mag = np.hypot(gx_, gy_)
    ang = (np.degrees(np.arctan2(gy_, gx_)) + 180) % 180
    return hysteresis(non_max_suppression(mag, ang), low, high)

for name, edges in [("Roberts (1963)", roberts_cross(noisy_cube) > 0.15), ("Canny (1986)", canny(noisy_cube))]:
    f1, p, r = edge_f1(edges, true_edges)
    print(f"{name:15s} on the noisy cube: F1 {f1:.2f}  (precision {p:.2f}, recall {r:.2f})")

fig, ax = plt.subplots(1, 3, figsize=(10.5, 3.5))
ax[0].imshow(noisy_cube, cmap="gray"); ax[0].set_title("noisy input")
ax[1].imshow(roberts_cross(noisy_cube) > 0.15, cmap="gray"); ax[1].set_title("Roberts")
ax[2].imshow(canny(noisy_cube), cmap="gray"); ax[2].set_title("Canny")
for a in ax: a.axis("off")
plt.show()

**What to notice:** each step fixes a specific problem. Smoothing removes most noise responses; non-maximum suppression thins edges to one pixel (precision); hysteresis keeps edges continuous (recall).

**✍️ Try it:** change `sigma` to 0.5 or 3, or the thresholds, and watch precision and recall trade off.

But even perfect edges don't say *what* an object is: a cat's outline in a cluttered photo is broken, mixed with background edges, and changes with viewpoint. **David Lowe (1987)** went further, grouping edges by **perceptual organization** (parallel or collinear lines) to recognize 3-D objects such as razors in cluttered scenes.

### 3.7 The AI winter, and what neuroscience was learning
By the late 1980s, enthusiasm and funding for AI **dwindled** ("expert systems" failed to deliver). Subfields like computer vision, NLP, robotics, and computational biology kept growing. **[CS231n 2026]** Meanwhile, cognitive science and neuroscience learned key facts about *human* vision:
* **Biederman (1972):** objects are recognized faster in **coherent scenes** than in jumbled ones: **context** matters.
* **Potter et al. (1970s):** people grasp the gist of images flashed in a rapid sequence (**rapid serial visual presentation**).
* **Thorpe et al. (1996):** brain signals distinguish "animal" vs. "no animal" photos about **150 ms** after they appear: recognition is extremely fast, suggesting a mostly **feed-forward** hierarchy (one sweep through the layers, like a CNN).
* **Kanwisher et al. (1997); Epstein & Kanwisher (1998):** dedicated brain regions for **faces** and for **places**.

Conclusion: **visual recognition is a fundamental task** for visual intelligence.

### 3.8 Recognition via grouping (1990s): normalized cuts
Before recognizing objects, first **group pixels into regions** that belong together. **Shi & Malik (1997)** turned this into a **graph problem**:
* Every **pixel is a node**.
* Every pair of nearby pixels gets an **edge weight** $w_{ij}$ (an **affinity**): large if the pixels are **close together and similar in brightness**, near zero otherwise:
$$w_{ij} = \exp\left(-\frac{(I_i - I_j)^2}{\sigma_I^2}\right)\exp\left(-\frac{\|\mathbf{x}_i - \mathbf{x}_j\|^2}{\sigma_X^2}\right)\quad\text{if }\|\mathbf{x}_i - \mathbf{x}_j\| < r,\ \text{else }0$$
* Splitting the pixels into two groups $A$ and $B$ **cuts** the edges between them; $\text{cut}(A, B) = \sum_{i\in A, j\in B}w_{ij}$. A good segmentation cuts only weak connections.

Plain "smallest cut" likes to slice off a few isolated pixels, so they **normalize** by each group's total connections, $\text{assoc}(A, V) = \sum_{i\in A, j}w_{ij}$:
$$\text{Ncut}(A, B) = \frac{\text{cut}(A, B)}{\text{assoc}(A, V)} + \frac{\text{cut}(A, B)}{\text{assoc}(B, V)}$$
Finding the exact best split is too slow, but an approximation reduces to an **eigenvector** problem (below).

**Tiny worked example: 4 "pixels"** in a row with brightness [0.1, 0.12, 0.9, 0.88]. Pixels 0–1 are similar, 2–3 are similar, and the two pairs are very different.

In [ ]:
values = np.array([0.1, 0.12, 0.9, 0.88])
affinity_tiny = np.exp(-(values[:, None] - values[None, :]) ** 2 / 0.1 ** 2)   # brightness similarity only, for clarity
print("affinity matrix W (rounded):\n", affinity_tiny.round(3))

def ncut_value(W, in_A):
    # Normalized cut of splitting the nodes into A (in_A True) and B (in_A False).
    d = W.sum(1)                                   # each node's total connection strength
    cut = W[in_A][:, ~in_A].sum()                  # weight of the edges we cut
    return cut / d[in_A].sum() + cut / d[~in_A].sum()

for split in [[True, True, False, False], [True, False, False, False], [True, False, True, False]]:
    print(f"split A = {np.nonzero(split)[0].tolist()}: Ncut = {ncut_value(affinity_tiny, np.array(split)):.4f}")

The natural split {0, 1} vs. {2, 3} has an Ncut near 0 (it cuts only near-zero weights); the bad splits are large. ✔

**The eigenvector shortcut.** Let $D$ be the diagonal matrix of each node's total connections, $D_{ii} = \sum_jw_{ij}$. Shi & Malik showed that a good approximate split comes from the eigenvector $y$ with the **second-smallest** eigenvalue of
$$(D - W)\,y = \lambda D\,y$$
(The smallest eigenvalue is always 0, with a constant eigenvector, which doesn't split anything.) Pixels with large $y$ go in one group, small $y$ in the other; we try several cut-off values and keep the one with the lowest Ncut.

**Step 1: a test image** with an elliptical object, **uneven lighting** (brightness increases from left to right), and noise.

In [ ]:
S_img = 28
gy_, gx_ = np.mgrid[0:S_img, 0:S_img]
truth = ((gx_ - 16) ** 2 / 64 + (gy_ - 13) ** 2 / 36 < 1)              # the true object: an ellipse
ncut_rng = np.random.default_rng(1)
image = (np.where(truth, 0.65, 0.35)                                    # object brighter than background...
         + 0.6 * gx_ / (S_img - 1)                                      # ...but lighting gets brighter to the right
         + ncut_rng.normal(0, 0.04, (S_img, S_img)))                    # ...plus noise
plt.figure(figsize=(3, 3)); plt.imshow(image, cmap="gray"); plt.title("unevenly lit object"); plt.axis("off"); plt.show()

**Step 2: build the affinity matrix** for all $28\times28 = 784$ pixels (a 784×784 matrix).

In [ ]:
def pixel_affinities(img, sigma_i=0.1, sigma_x=4.0, radius=5):
    # Affinity w_ij between every pair of pixels: similar brightness AND close together -> near 1; otherwise near 0.
    rows, cols = np.mgrid[0:img.shape[0], 0:img.shape[1]]
    coords = np.c_[rows.ravel(), cols.ravel()].astype(float)            # (pixel, 2) positions
    vals = img.ravel()
    dist2 = ((coords[:, None, :] - coords[None, :, :]) ** 2).sum(-1)     # squared distance between every pair
    bright2 = (vals[:, None] - vals[None, :]) ** 2                      # squared brightness difference
    W = np.exp(-bright2 / sigma_i ** 2) * np.exp(-dist2 / sigma_x ** 2)
    return W * (dist2 < radius ** 2)                                    # only connect nearby pixels

W_img = pixel_affinities(image)
print("affinity matrix shape:", W_img.shape, "  fraction of nonzero entries:", round(float(np.mean(W_img > 0)), 3))

**Step 3: solve the eigenproblem** and look at the second eigenvector, reshaped back into an image.

In [ ]:
D_img = np.diag(W_img.sum(1))
eigvals, eigvecs = eigh(D_img - W_img, D_img)            # generalized eigenproblem (D - W) y = lambda D y
print("three smallest eigenvalues:", eigvals[:3].round(5), " (the first is ~0)")
fiedler = eigvecs[:, 1]                                 # the 2nd-smallest eigenvector (called the 'Fiedler vector')
plt.figure(figsize=(3, 3)); plt.imshow(fiedler.reshape(S_img, S_img), cmap="RdBu"); plt.title("2nd eigenvector"); plt.axis("off"); plt.show()

**Step 4: choose the cut-off** that gives the lowest Ncut, and compare with simply thresholding brightness. Since the two groups' names are arbitrary, we score agreement either way round.

In [ ]:
candidates = np.quantile(fiedler, np.linspace(0.02, 0.98, 49))           # 49 possible cut-off values
best_threshold = min(candidates, key=lambda t: ncut_value(W_img, fiedler > t))
segmentation = (fiedler > best_threshold).reshape(S_img, S_img)

agree = lambda mask: max(np.mean(mask == truth), np.mean(mask != truth))   # fraction of pixels labeled correctly
best_single_threshold = max(agree(image > t) for t in np.linspace(image.min(), image.max(), 200))
print(f"normalized cut:                         {agree(segmentation):.1%} of pixels correct")
print(f"threshold at the mean brightness:        {agree(image > image.mean()):.1%}")
print(f"best possible single brightness cut-off: {best_single_threshold:.1%}  (chosen by peeking at the answer!)")

fig, ax = plt.subplots(1, 2, figsize=(6, 3))
ax[0].imshow(segmentation, cmap="gray"); ax[0].set_title("normalized cut")
ax[1].imshow(image > image.mean(), cmap="gray"); ax[1].set_title("brightness > mean")
for a in ax: a.axis("off")
plt.show()

**What to notice:** because of the lighting, the background on the right is as bright as the object on the left, so **no single brightness threshold works**. Grouping uses a different, more robust cue: *nearby pixels with similar brightness belong together*, which survives slow lighting changes. (Its limit: with heavy noise, say 0.12 instead of 0.04, the eigenvector approximation breaks down. ✍️ Try it.) Normalized cuts is also the origin of **spectral clustering**, still widely used. But grouping only finds regions, not *which object* each region is.

### 3.9 Recognition via matching (2000s): SIFT
**David Lowe's SIFT** (*Scale-Invariant Feature Transform*, 1999) recognizes an object by **matching distinctive local patches** between a reference image and a new image. The key: describe each patch in a way that stays the same when the image is rotated, brightened, or slightly changed. Three stages:
1. **Detect** keypoints: points that can be found again reliably (we'll use corners).
2. **Describe** each keypoint with a vector that ignores rotation and lighting.
3. **Match** descriptors between images, rejecting ambiguous matches.

**Step 1: test images.** Image 1 is a random texture; image 2 is the same texture **rotated 30°, brighter, higher-contrast, and noisy**. Because we made image 2 ourselves, we know exactly where every point of image 1 went, so we can check every match.

In [ ]:
def textured_image(size=160, seed=3):
    # A random texture: smooth blotches plus a few small bright spots, scaled to [0, 1].
    r = np.random.default_rng(seed)
    im = ndimage.gaussian_filter(r.random((size, size)), 2.5)
    im += 0.5 * ndimage.gaussian_filter((r.random((size, size)) > 0.985).astype(float), 1.5)
    return (im - im.min()) / (im.max() - im.min())

img1 = textured_image()
angle = 30
sift_rng = np.random.default_rng(2)
img2 = np.clip(1.3 * ndimage.rotate(img1, angle, reshape=False, order=1) + 0.1    # rotate, more contrast, brighter
               + sift_rng.normal(0, 0.02, img1.shape), 0, 1.6)                   # plus noise

def true_location_in_img2(pt):
    # Where a point (row, col) of img1 ends up in img2 (rotation by `angle` about the image center).
    center = np.array(img1.shape) / 2 - 0.5
    th = np.radians(angle); r, c = pt - center
    return np.array([-c * np.sin(th) + r * np.cos(th), c * np.cos(th) + r * np.sin(th)]) + center

fig, ax = plt.subplots(1, 2, figsize=(7, 3.5))
ax[0].imshow(img1, cmap="gray"); ax[0].set_title("image 1"); ax[1].imshow(img2, cmap="gray"); ax[1].set_title("image 2: rotated, relit, noisy")
for a in ax: a.axis("off")
plt.show()

**Step 2: detect corners (🔎 the Harris detector, 1988).** A corner is a point where brightness changes in **two** directions (an edge changes in only one; a flat region in none). Harris summarizes the gradients around each pixel in a 2×2 matrix, the **structure tensor**:
$$M = \sum_{\text{window}}\begin{bmatrix}g_x^2 & g_xg_y\\ g_xg_y & g_y^2\end{bmatrix},\qquad \text{corner score } R = \det(M) - k\,(\text{trace } M)^2$$
* **Flat** region: both eigenvalues of $M$ small → $R\approx0$.
* **Edge:** one eigenvalue large, one small → $R < 0$.
* **Corner:** both large → $R$ large and positive.

Worked example on three tiny patches:

In [ ]:
def harris_response(I, k=0.05, sigma=1.5):
    # Harris corner score R = det(M) - k * trace(M)^2 at every pixel (M averaged over a Gaussian window of width sigma).
    Ix, Iy = ndimage.sobel(I, 1), ndimage.sobel(I, 0)
    Sxx = ndimage.gaussian_filter(Ix * Ix, sigma)
    Syy = ndimage.gaussian_filter(Iy * Iy, sigma)
    Sxy = ndimage.gaussian_filter(Ix * Iy, sigma)
    return Sxx * Syy - Sxy ** 2 - k * (Sxx + Syy) ** 2

flat = np.zeros((15, 15)); edge = np.zeros((15, 15)); edge[:, 8:] = 1; corner = np.zeros((15, 15)); corner[8:, 8:] = 1
for name, patch in [("flat", flat), ("edge", edge), ("corner", corner)]:
    print(f"{name:6s} patch: Harris score at the center = {harris_response(patch)[7, 7]:8.2f}")

Now keep the 60 strongest corners in each image. A pixel counts as a corner peak if it's the maximum in its 7×7 neighborhood; we ignore a 20-pixel border so every keypoint has room for its descriptor patch.

In [ ]:
def harris_corners(I, n=60, border=20):
    # The n strongest corner peaks of image I, as an array of (row, col).
    R = harris_response(I)
    R[:border] = R[-border:] = 0; R[:, :border] = R[:, -border:] = 0          # stay away from the edges
    is_peak = (R == ndimage.maximum_filter(R, 7)) & (R > 0)                  # local maxima with positive score
    rows, cols = np.nonzero(is_peak)
    strongest = np.argsort(-R[rows, cols])[:n]
    return np.c_[rows[strongest], cols[strongest]]

kp1, kp2 = harris_corners(img1), harris_corners(img2)
print(f"keypoints: {len(kp1)} in image 1, {len(kp2)} in image 2")

**Step 3: describe each keypoint.** A SIFT-style descriptor in three parts:
1. **Dominant orientation:** find the most common gradient direction around the keypoint, and **rotate the patch** so that direction points the same way in every image. → *rotation invariance*.
2. **Histograms of gradient orientations:** split the rotated 16×16 patch into a 4×4 grid of cells; in each cell, count gradient directions in 8 bins, weighted by gradient strength. 4×4×8 = **128 numbers**. Gradients ignore uniform brightness changes, and histograms tolerate small shifts.
3. **Normalize** the vector to length 1, cap large values at 0.2, and normalize again. → *robustness to contrast changes*.

We write each part as its own small function.

In [ ]:
def dominant_orientation(patch):
    # Most common gradient direction (degrees) in the central part of `patch`, weighted by gradient strength.
    gy, gx = np.gradient(ndimage.gaussian_filter(patch, 1.0))
    angles, strength = np.degrees(np.arctan2(gy, gx)), np.hypot(gx, gy)
    q = patch.shape[0] // 4
    hist, edges = np.histogram(angles[q:-q, q:-q], bins=36, range=(-180, 180), weights=strength[q:-q, q:-q])
    return edges[np.argmax(hist)] + 5                     # center of the winning 10-degree bin

def orientation_histograms(patch16):
    # 4x4 grid of cells over a 16x16 patch; 8-bin gradient-orientation histogram per cell -> 128 numbers.
    gy, gx = np.gradient(ndimage.gaussian_filter(patch16, 1.0))
    angles, strength = (np.degrees(np.arctan2(gy, gx)) + 360) % 360, np.hypot(gx, gy)
    feats = []
    for i in range(4):
        for j in range(4):
            cell = (slice(4 * i, 4 * i + 4), slice(4 * j, 4 * j + 4))
            feats.extend(np.histogram(angles[cell], bins=8, range=(0, 360), weights=strength[cell])[0])
    return np.array(feats)

def normalize_descriptor(d):
    # Unit length, cap at 0.2 (so a few huge gradients can't dominate), unit length again.
    d = d / (np.linalg.norm(d) + 1e-12)
    d = np.minimum(d, 0.2)
    return d / (np.linalg.norm(d) + 1e-12)

def describe(I, row, col, half=16):
    # SIFT-style 128-D descriptor of the keypoint at (row, col).
    big = I[row - half:row + half, col - half:col + half]                  # 32x32 patch (room to rotate)
    upright = ndimage.rotate(big, dominant_orientation(big), reshape=False, order=1)   # rotate to canonical orientation
    core = upright[half // 2:half // 2 * 3, half // 2:half // 2 * 3]       # central 16x16
    return normalize_descriptor(orientation_histograms(core))

D1 = np.array([describe(img1, *p) for p in kp1]); D2 = np.array([describe(img2, *p) for p in kp2])
print("descriptor matrix shapes:", D1.shape, D2.shape, " (one 128-D row per keypoint)")

**Step 4: match, with Lowe's ratio test.** For each descriptor in image 1, find its **nearest** and **second-nearest** descriptors in image 2. If the nearest is not clearly better than the runner-up ($d_1/d_2 < 0.8$), the match is **ambiguous** (several patches look alike), so we drop it.

Worked example: a query with distances 0.3 to its best candidate and 0.9 to the next (ratio 0.33: confident), vs. 0.5 and 0.55 (ratio 0.91: ambiguous).

In [ ]:
for d_best, d_second in [(0.3, 0.9), (0.5, 0.55)]:
    ratio = d_best / d_second
    print(f"distances {d_best} vs {d_second}: ratio {ratio:.2f} -> {'keep' if ratio < 0.8 else 'reject (ambiguous)'}")

def match(D_a, D_b, use_ratio_test=True, ratio=0.8):
    # List of (i, j) pairs: descriptor i of image A matched to descriptor j of image B.
    dist = np.linalg.norm(D_a[:, None] - D_b[None], axis=2)              # every pairwise distance
    pairs = []
    for i in range(len(D_a)):
        j1, j2 = np.argsort(dist[i])[:2]                                   # nearest and second nearest
        if not use_ratio_test or dist[i, j1] < ratio * dist[i, j2]:
            pairs.append((i, j1))
    return pairs

**Step 5: evaluate.** A match is correct if image 2's keypoint is within 4 pixels of where the image-1 keypoint truly moved. As a baseline we also try matching **raw pixel patches** (normalized for brightness, but not rotated).

In [ ]:
def fraction_correct(pairs, k_a, k_b):
    return np.mean([np.linalg.norm(true_location_in_img2(k_a[i]) - k_b[j]) < 4 for i, j in pairs]) if pairs else 0.0

for name, use_ratio in [("SIFT-style, nearest neighbor", False), ("SIFT-style + ratio test", True)]:
    pairs = match(D1, D2, use_ratio)
    print(f"{name:30s}: {len(pairs):2d} matches, {fraction_correct(pairs, kp1, kp2):.0%} correct")

def raw_patch(I, p):
    v = I[p[0] - 8:p[0] + 8, p[1] - 8:p[1] + 8].ravel(); return (v - v.mean()) / (v.std() + 1e-9)
P1, P2 = np.array([raw_patch(img1, p) for p in kp1]), np.array([raw_patch(img2, p) for p in kp2])
raw_pairs = match(P1, P2, use_ratio_test=False)
print(f"{'raw pixel patches':30s}: {len(raw_pairs):2d} matches, {fraction_correct(raw_pairs, kp1, kp2):.0%} correct")

good_pairs = match(D1, D2, True)
plt.figure(figsize=(9, 4.5)); plt.imshow(np.hstack([img1, img2 / img2.max()]), cmap="gray")
for i, j in good_pairs:
    ok = np.linalg.norm(true_location_in_img2(kp1[i]) - kp2[j]) < 4
    plt.plot([kp1[i][1], kp2[j][1] + img1.shape[1]], [kp1[i][0], kp2[j][0]], color="lime" if ok else "red", lw=0.8)
plt.title("matches after the ratio test (green = correct)"); plt.axis("off"); plt.show()

**What to notice:**
* Raw pixels fail: once the image is rotated, the patches don't line up pixel by pixel.
* The rotation-normalized orientation histograms match much better, and the **ratio test** removes almost all the wrong matches (fewer matches, but trustworthy ones).

SIFT powered panorama stitching, 3-D reconstruction from photo collections, and object recognition through the 2000s. **"Bag of visual words"** classifiers (histograms of quantized SIFT descriptors fed to a linear classifier) were the state of the art in image classification until 2012 (Lecture 5).

### 3.10 Face detection: Viola & Jones (2001)
One of the **first big successes of machine learning in vision**: a face detector fast enough for real time, soon built into digital cameras. A detector checks hundreds of thousands of windows per image, so every window must be classified extremely cheaply. Three ideas made that possible:
1. **Haar-like features:** the sum of pixels in one rectangle minus the sum in an adjacent one ("is the eye region darker than the cheeks?").
2. **The integral image**, which makes any rectangle sum cost 4 lookups.
3. **AdaBoost**, which picks the few most useful features out of thousands, and a **cascade** that rejects most windows after one or two features.

**Step 1: the integral image.** $II(y, x)$ = the sum of all pixels above and to the left of $(y, x)$. Then the sum over any rectangle is
$$\sum_{\text{rectangle}} = II(\text{bottom-right}) - II(\text{top-right}) - II(\text{bottom-left}) + II(\text{top-left})$$
(adding back the top-left corner, which was subtracted twice). Worked example on a 3×3 image:

In [ ]:
def integral_image(I):
    # Cumulative sums over rows and columns, with a row and column of zeros in front.
    # Works for one image (H, W) or a stack of images (N, H, W).
    pad = [(0, 0)] * (I.ndim - 2) + [(1, 0), (1, 0)]
    return np.pad(I, pad).cumsum(-2).cumsum(-1)

def rect_sum(II, top, left, bottom, right):
    # Sum of I[top:bottom, left:right] using only 4 lookups into the integral image.
    return II[..., bottom, right] - II[..., top, right] - II[..., bottom, left] + II[..., top, left]

tiny = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]], float)
II_tiny = integral_image(tiny)
print("image:\n", tiny); print("integral image (with the zero row/column):\n", II_tiny)
print("sum of the bottom-right 2x2 block [[5, 6], [8, 9]] via 4 lookups:", rect_sum(II_tiny, 1, 1, 3, 3), "(direct: 28)")

**Step 2: training data.** Real Viola–Jones learned from thousands of face photos. We use cartoon 16×16 "faces" (two dark eyes, a darker mouth, random small shifts, lighting changes, and noise) vs. "non-faces" (random smooth textures).

In [ ]:
P = 16                                                    # window size
def make_face(r):
    # A cartoon 16x16 face: dark eyes and mouth on a lighter background, randomly shifted, relit, and noisy.
    f = np.full((P, P), 0.6)
    f[4:7, 3:7] = f[4:7, 9:13] = 0.2                      # two dark eyes
    f[11:13, 5:11] = 0.3                                  # a darker mouth
    f = np.roll(f, (r.integers(-1, 2), r.integers(-1, 2)), (0, 1))   # shift by up to 1 pixel
    return f * r.uniform(0.7, 1.3) + r.normal(0, 0.08, (P, P))       # lighting + noise

def make_nonface(r):
    # A random smooth texture with random brightness.
    g = ndimage.gaussian_filter(r.random((P, P)), r.uniform(0.5, 3))
    g = (g - g.min()) / (g.max() - g.min() + 1e-9)
    return g * r.uniform(0.3, 1.0) + r.normal(0, 0.08, (P, P))

vj_rng = np.random.default_rng(1)
X = np.array([make_face(vj_rng) for _ in range(1000)] + [make_nonface(vj_rng) for _ in range(1000)])
y = np.r_[np.ones(1000), -np.ones(1000)]                 # +1 = face, -1 = non-face
order = vj_rng.permutation(2000); X, y = X[order], y[order]
IIs = integral_image(X)                                    # integral images of all 2000 windows at once
train, test = slice(0, 1400), slice(1400, None)

fig, ax = plt.subplots(1, 4, figsize=(7, 2))
for a, im, t in zip(ax, [make_face(vj_rng), make_face(vj_rng), make_nonface(vj_rng), make_nonface(vj_rng)], ["face", "face", "non-face", "non-face"]):
    a.imshow(im, cmap="gray"); a.set_title(t, fontsize=8); a.axis("off")
plt.show()

**Step 3: Haar-like features.** Each feature is two side-by-side rectangles (top/bottom or left/right) of random size and position; its value is (sum of the first) − (sum of the second). We generate 600 random features and compute all of them for all windows; thanks to the integral image, each costs only 8 lookups.

In [ ]:
def random_haar_features(n, r):
    # n random two-rectangle features, each (kind, top, left, height, width). kind 0 = top/bottom, 1 = left/right.
    feats = []
    for _ in range(n):
        h, w = r.integers(2, 7), r.integers(2, 7)
        top, left = r.integers(0, P - 2 * h + 1), r.integers(0, P - 2 * w + 1)
        feats.append((r.integers(2), top, left, h, w))
    return feats

def haar_value(IIs, feat):
    # Value of one Haar feature for every window: (first rectangle sum) - (second rectangle sum).
    kind, top, left, h, w = feat
    if kind == 0:   # upper rectangle minus the one below it
        return rect_sum(IIs, top, left, top + h, left + w) - rect_sum(IIs, top + h, left, top + 2 * h, left + w)
    return rect_sum(IIs, top, left, top + h, left + w) - rect_sum(IIs, top, left + w, top + h, left + 2 * w)

features = random_haar_features(600, vj_rng)
F = np.stack([haar_value(IIs, f) for f in features], axis=1)     # (2000 windows, 600 features)
print("feature matrix:", F.shape)

**Step 4: AdaBoost.** A **weak classifier** (a "stump") looks at **one feature** and a threshold: "face if feature > threshold" (or <). Each one alone is barely better than guessing. AdaBoost combines them:
1. Give every training example a **weight** (all equal at first).
2. Pick the stump with the lowest **weighted** error $\epsilon$.
3. Give it a vote $\alpha = \tfrac12\ln\frac{1-\epsilon}{\epsilon}$ (more accurate stumps get bigger votes).
4. **Increase the weights of the examples it got wrong** (by $e^{\alpha}$) and decrease the others, so the next stump focuses on the hard cases.
5. Repeat. The final classifier is the sign of the weighted vote $\sum_t\alpha_th_t(x)$.

First, the stump search on its own:

In [ ]:
def best_stump(F, y, w):
    # Search all features, 19 thresholds each, and both directions. Return (weighted error, feature, threshold, polarity).
    best = None
    for j in range(F.shape[1]):
        for thr in np.quantile(F[:, j], np.linspace(0.05, 0.95, 19)):
            for polarity in (1, -1):
                pred = np.where(polarity * (F[:, j] - thr) > 0, 1, -1)
                err = w[pred != y].sum()                      # weighted error
                if best is None or err < best[0]:
                    best = (err, j, thr, polarity)
    return best

w0 = np.ones(1400) / 1400
err, j, thr, pol = best_stump(F[train], y[train], w0)
print(f"best single stump: feature {j}, threshold {thr:.2f}, training error {err:.1%}")

Now the full boosting loop, which repeats the stump search with re-weighted examples:

In [ ]:
def adaboost(F, y, rounds):
    # Train `rounds` boosted stumps. Returns a list of (alpha, feature, threshold, polarity).
    w = np.ones(len(y)) / len(y)
    model = []
    for _ in range(rounds):
        err, j, thr, pol = best_stump(F, y, w)
        err = max(err, 1e-10)
        alpha = 0.5 * np.log((1 - err) / err)                     # this stump's vote
        pred = np.where(pol * (F[:, j] - thr) > 0, 1, -1)
        w *= np.exp(-alpha * y * pred)                            # wrong (y*pred = -1) -> weight grows
        w /= w.sum()
        model.append((alpha, j, thr, pol))
    return model

def boosted_score(model, F):
    # Weighted vote of the stumps; positive = face.
    return sum(a * np.where(p * (F[:, j] - t) > 0, 1, -1) for a, j, t, p in model)

t0 = time.time()
model = adaboost(F[train], y[train], 10)
for k in [1, 2, 5, 10]:
    acc = np.mean(np.sign(boosted_score(model[:k], F[test])) == y[test])
    print(f"{k:2d} features: test accuracy {acc:.1%}")
print(f"({time.time() - t0:.0f}s)")

**Step 5: the cascade.** Most windows in a photo are obviously not faces, so a cheap **first stage** uses just the first few features, with its threshold set to keep 99% of faces; anything it rejects is discarded immediately, and only survivors go on to more expensive stages.

One subtlety: with only 2 stumps, the boosted score can take just a handful of distinct values (each stump votes ±α), so "the threshold that keeps 99% of faces" may land on the lowest possible score and reject nothing. With 3 stumps there are enough distinct scores. We print both.

In [ ]:
for k in [2, 3]:
    stage1 = boosted_score(model[:k], F[test])
    threshold = np.quantile(stage1[y[test] == 1], 0.01)          # threshold that keeps ~99% of the faces
    kept = np.mean(stage1[y[test] == 1] >= threshold)
    rejected = np.mean(stage1[y[test] == -1] < threshold)
    print(f"stage 1 with {k} features: {len(np.unique(stage1))} distinct score values; keeps {kept:.1%} of faces, "
          f"immediately rejects {rejected:.0%} of non-faces")

fig, ax = plt.subplots(1, 2, figsize=(4.5, 2.4))
for k in range(2):
    _, j, _, _ = model[k]; kind, top, left, h, w = features[j]; vis = np.zeros((P, P))
    if kind == 0: vis[top:top + h, left:left + w], vis[top + h:top + 2 * h, left:left + w] = 1, -1
    else: vis[top:top + h, left:left + w], vis[top:top + h, left + w:left + 2 * w] = 1, -1
    ax[k].imshow(vis, cmap="RdBu", vmin=-1, vmax=1); ax[k].set_title(f"feature {k + 1}", fontsize=8); ax[k].axis("off")
plt.suptitle("the first two features AdaBoost picked", fontsize=8); plt.show()

**What to notice:** with only a handful of features, boosting classifies nearly perfectly, and the features it picks line up with face structure (eyes vs. cheeks). A 3-feature first stage already throws away most non-faces at almost no cost, which is what made real-time detection possible.

**The lesson of §3:** each era **hand-designed** a representation (edges, regions, local descriptors, rectangle features), and things improved once **machine learning chose how to combine features** (Viola–Jones; SIFT + classifiers). The next step was to **learn the features themselves**, which needed data, compute, and an algorithm (§4–5).

---
## 4. Datasets and the ImageNet challenge

Learning features needs lots of labeled data. Benchmarks grew fast (sizes from the CS231n 2026 slides):

| Dataset | Year | Size |
|---|---|---|
| Caltech101 (Fei-Fei, Fergus & Perona) | 2003–04 | ~9K images, 101 categories |
| PASCAL Visual Object Challenge (Everingham et al.) | 2005–12 | ~30K images, 20 categories |
| LabelMe (Russell et al.) | 2007 | ~37K images |
| SUN (Xiao et al.) | 2010 | ~131K images |
| **ImageNet** (Deng et al., Fei-Fei, CVPR 2009) | 2009 | **~15M images, ~22K categories** |

**The ImageNet Large Scale Visual Recognition Challenge (ILSVRC)**, 2010–2017: **1,000 classes, 1,431,167 images**. The score is **top-5 error**: a prediction counts as correct if the true label is among the model's 5 guesses (photos often contain several objects). The 2009 ImageNet paper later won a prize for long-term impact (IEEE PAMI Longuet-Higgins Prize, 2019).

The winners' error rates:

In [ ]:
ilsvrc = [(2010, 28.2, "Lin et al."), (2011, 25.8, "Sanchez & Perronnin"), (2012, 16.4, "AlexNet"),
          (2013, 11.7, "ZFNet"), (2014, 6.7, "GoogLeNet"), (2015, 3.6, "ResNet"),
          (2016, 3.0, "Trimps-Soushen"), (2017, 2.3, "SENet")]
years, errors, names = zip(*ilsvrc)
plt.figure(figsize=(8, 4))
plt.bar(years, errors, color=["gray", "gray"] + ["tab:blue"] * 6)       # gray = hand-designed features, blue = deep nets
plt.axhline(5.1, color="red", ls="--", label="a trained human (~5.1%)")
for yr, e, nm in ilsvrc:
    plt.text(yr, e + 0.6, f"{e}\n{nm}", ha="center", fontsize=7)
plt.ylabel("top-5 error (%)"); plt.title("ImageNet winners (gray: hand-designed features; blue: deep networks)"); plt.legend(); plt.ylim(0, 33); plt.show()
print(f"2011 -> 2012 improvement: {25.8 - 16.4:.1f} points in one year")

**What to notice:** before 2012, winners used hand-designed features plus linear classifiers, gaining a few points a year. In **2012, AlexNet**, a deep convolutional network trained on GPUs, cut the error from 25.8% to 16.4%. Every winner afterward was a deep network, and by 2015 they beat the ~5% error of a trained human annotator (measured by Andrej Karpathy on a subset; label noise makes the last fraction of a percent hard to interpret). **[CS231n 2026]** shows the same trend for top-1 accuracy, past 90% for the best models in the 2020s.

---
## 5. A brief history of deep learning

### 5.1 The perceptron (Frank Rosenblatt, ~1957–58)
One of the first learning machines, built in hardware: the **Mark I Perceptron** connected a **20×20 grid of light sensors** (a 400-pixel camera) to adjustable weights (motor-driven potentiometers). It computes
$$f(x) = \text{sign}(w^\top x + b)\in\{-1, +1\}$$
In modern terms it's a **linear classifier** (Lecture 3). **The learning rule:** go through the examples; whenever one is misclassified, nudge the weights toward the right answer:
$$w\leftarrow w + y\,x,\qquad b\leftarrow b + y$$
(If a positive example ($y = +1$) scored too low, adding $x$ raises its score next time; for a negative one, subtracting $x$ lowers it.)

**Worked example, traced step by step:** four 2-D points, two of each class.

In [ ]:
X_toy = np.array([[2.0, 1.0], [1.0, 3.0], [-1.0, -1.0], [-2.0, 1.0]])
y_toy = np.array([1, 1, -1, -1])
w, b = np.zeros(2), 0.0
for epoch in range(3):
    for x_i, y_i in zip(X_toy, y_toy):
        score = w @ x_i + b
        if y_i * score <= 0:                         # wrong side (or on the boundary): update
            w, b = w + y_i * x_i, b + y_i
            print(f"epoch {epoch}: x = {x_i}, y = {y_i:+d}, score {score:+.1f} -> MISTAKE, now w = {w}, b = {b:+.0f}")
        else:
            print(f"epoch {epoch}: x = {x_i}, y = {y_i:+d}, score {score:+.1f} -> correct")
print("final scores:", X_toy @ w + b, "-> signs", np.sign(X_toy @ w + b), "match labels", y_toy)

After a single mistake (on the very first point, when all weights were zero), every point is on the correct side. Now as a reusable function, applied to 20×20 "light-sensor" images of the letters T and L:

In [ ]:
def perceptron(X, y, epochs=200):
    # Rosenblatt's perceptron. X: (N, D) inputs, y: (N,) labels in {-1, +1}.
    # Returns (weights, bias, epochs used); epochs used is None if it never got everything right.
    w, b = np.zeros(X.shape[1]), 0.0
    for epoch in range(epochs):
        mistakes = 0
        for x_i, y_i in zip(X, y):
            if y_i * (w @ x_i + b) <= 0:
                w += y_i * x_i; b += y_i; mistakes += 1
        if mistakes == 0:
            return w, b, epoch + 1
    return w, b, None

def letter_image(kind, r):
    # A 20x20 image of the letter T or L, plus noise.
    im = np.zeros((20, 20))
    if kind == "T": im[4, 4:16] = 1; im[4:16, 9:11] = 1
    else:           im[4:16, 4] = 1; im[15, 4:16] = 1
    return np.clip(im + r.normal(0, 0.3, im.shape), 0, 1)

p_rng = np.random.default_rng(0)
X_letters = np.array([letter_image(k, p_rng).ravel() for k in ["T", "L"] * 100])   # 200 images, flattened to 400 numbers
y_letters = np.array([1, -1] * 100)                                              # T = +1, L = -1
w, b, epochs = perceptron(X_letters[:150], y_letters[:150])
print(f"T vs L: learned in {epochs} epochs; accuracy on 50 new images {np.mean(np.sign(X_letters[150:] @ w + b) == y_letters[150:]):.0%}")

Learning worked, and caused the first wave of AI hype (the *New York Times* in 1958 reported the Navy expected such a machine to "walk, talk, see, write, reproduce itself and be conscious of its existence").

### 5.2 Minsky and Papert (1969): what perceptrons can't do
Their book *Perceptrons* proved that a single-layer perceptron **cannot learn some simple functions**. The famous one is **XOR**:

| $x$ | $y$ | XOR |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

No single straight line separates {(0,1), (1,0)} from {(0,0), (1,1)}. This caused a lot of disillusionment in the field.

A version that matters for vision: **can a perceptron tell a vertical full-length line from a horizontal one, anywhere in the image?** No, and the proof is two lines:
* If every **vertical** line (column $c$) scores positive: $\sum_rw_{rc} > -b$ for every $c$. Add these up over all 20 columns: $\sum_{r,c}w_{rc} > -20b$.
* If every **horizontal** line (row $r$) scores negative: $\sum_cw_{rc} < -b$ for every $r$. Add up over the 20 rows: $\sum_{r,c}w_{rc} < -20b$.
* The same total can't be both bigger and smaller than $-20b$. **Contradiction.**

Let's watch the perceptron fail on both:

In [ ]:
def line_image(vertical, pos):
    # A 20x20 image with one full-length line: column `pos` if vertical, else row `pos`.
    im = np.zeros((20, 20))
    if vertical: im[:, pos] = 1
    else: im[pos, :] = 1
    return im.ravel()

X_lines = np.array([line_image(v, p) for v in (True, False) for p in range(20)])
y_lines = np.array([1] * 20 + [-1] * 20)
w, b, epochs = perceptron(X_lines, y_lines, epochs=500)
print(f"vertical vs horizontal lines: converged? {epochs is not None}; accuracy on its own training data {np.mean(np.sign(X_lines @ w + b) == y_lines):.0%}")

X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float); y_xor = np.array([-1, 1, 1, -1])
_, _, epochs = perceptron(X_xor, y_xor, epochs=500)
print(f"XOR: converged? {epochs is not None}")

**What to notice:** the perceptron keeps cycling forever, exactly as the proof says. The missing ingredient is **structure**: local oriented detectors (Hubel & Wiesel's simple cells) followed by pooling over positions (complex cells) would solve the line task easily, since "the strongest vertical-edge response anywhere" is high only for vertical lines. That's the convolution + pooling recipe.

### 5.3 The Neocognitron (Kunihiko Fukushima, 1980)
A computational model of vision **directly inspired by Hubel and Wiesel**: layers of **simple cells (convolution)** alternating with **complex cells (pooling)**, stacked into a hierarchy. Remarkably close to modern CNNs, but with **no practical training algorithm**: it was trained layer by layer with hand-designed rules.

### 5.4 Backpropagation (Rumelhart, Hinton & Williams, 1986)
**Backpropagation** computes how much each weight in a multi-layer network contributed to the error, so all layers can be trained together (Lectures 5–6). With a hidden layer, XOR becomes easy. The network:
$$h = \tanh(xW_1 + b_1)\ \ (\text{hidden layer, 8 units}),\qquad p = \sigma(hW_2 + b_2)\ \ (\text{output, a probability})$$
**Step 1: set up the data and random starting weights.**

In [ ]:
X4 = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)
t4 = np.array([0, 1, 1, 0], float)                      # XOR targets
init_rng = np.random.default_rng(0)
W1, b1 = init_rng.normal(0, 1, (2, 8)), np.zeros(8)      # input (2) -> hidden (8)
W2, b2 = init_rng.normal(0, 1, (8, 1)), np.zeros(1)      # hidden (8) -> output (1)
sigmoid = lambda z: 1 / (1 + np.exp(-z))                 # squashes any number into (0, 1)

**Step 2: train.** Each step runs the **forward pass** (compute the prediction), then the **backward pass** (the chain rule, from the output back to each weight), then moves every weight a little against its gradient. The comments name each piece; Lecture 6 derives them.

In [ ]:
learning_rate = 1.0
for step in range(5000):
    # forward pass
    h = np.tanh(X4 @ W1 + b1)                    # hidden activations, shape (4, 8)
    p = sigmoid(h @ W2 + b2).ravel()             # predicted probability of "1", shape (4,)
    # backward pass (gradients of the average cross-entropy loss)
    d_out = (p - t4)[:, None] / 4                # d loss / d (output score)
    dW2, db2 = h.T @ d_out, d_out.sum(0)         # gradients for the output layer
    d_h = d_out @ W2.T * (1 - h ** 2)            # back through tanh (its derivative is 1 - tanh^2)
    dW1, db1 = X4.T @ d_h, d_h.sum(0)            # gradients for the hidden layer
    # gradient descent update
    W1 -= learning_rate * dW1; b1 -= learning_rate * db1
    W2 -= learning_rate * dW2; b2 -= learning_rate * db2
print("XOR predictions:", p.round(3), "  targets:", t4)

**What to notice:** the hidden layer learns intermediate features that make XOR linearly separable for the output layer. (With only 4 hidden units, some random starts get stuck with two outputs at 0.5, a **local minimum**. ✍️ Try `(2, 4)` and `(4, 1)` with seed 0. Making training reliable is a big part of Lectures 10–11.)

### 5.5 Convolutional networks: LeCun et al. (1998), LeNet
Applied backprop to a **Neocognitron-like architecture** (convolution + subsampling layers, then fully connected layers) and **learned to recognize handwritten digits**. It was deployed commercially to read checks (the slides credit NEC; most historical accounts describe an AT&T Bell Labs system deployed by NCR, which by the late 1990s read a significant share of US checks). **Very similar to our modern convolutional networks!**

### 5.6 2000s: "deep learning"
People tried to train **deeper and deeper** networks (Hinton & Salakhutdinov 2006; Bengio et al. 2007; Lee et al. 2009; Glorot & Bengio 2010, who analyzed why deep networks are hard to train). It was **not a mainstream research topic**, and there was **no good dataset**. ImageNet (2009) fixed that.

### 5.7 2012: AlexNet
Krizhevsky, Sutskever & Hinton (2012): a deep convolutional network (5 convolution layers + 3 fully connected), with ReLU activations, dropout, and data augmentation, trained on **two GTX 580 GPUs** (3 GB each) for **5–6 days**. Let's count what that network costs.

**The formulas** (Lecture 7 explains them fully). A **convolution layer** has $C_{\text{out}}$ filters, each of size $C_{\text{in}}\times k\times k$, plus one bias per filter. Each output value is one dot product of length $C_{\text{in}}k^2$:
$$\text{output size } H' = \left\lfloor\frac{H + 2p - k}{s}\right\rfloor + 1,\qquad \text{parameters} = C_{\text{out}}(C_{\text{in}}k^2 + 1),\qquad \text{multiply-adds} = H'W'C_{\text{out}}\cdot C_{\text{in}}k^2$$
($p$ = padding, $s$ = stride.) A **fully connected layer** from $n_{\text{in}}$ to $n_{\text{out}}$ has $n_{\text{out}}(n_{\text{in}} + 1)$ parameters and $n_{\text{in}}n_{\text{out}}$ multiply-adds. **Pooling** has no parameters.

**Step 1: the output-size formula, checked on AlexNet's first layer** (224×224 input, 11×11 filters, stride 4, padding 2):

In [ ]:
def conv_out(h, k, s, p):
    # Output height (or width) of a convolution or pooling layer: floor((h + 2p - k) / s) + 1.
    return (h + 2 * p - k) // s + 1

print("conv1 output size:", conv_out(224, 11, 4, 2), "  -> (224 + 4 - 11) // 4 + 1 = 55")
print("conv1 parameters :", 64 * (3 * 11 * 11 + 1), "  -> 64 filters x (3*11*11 weights + 1 bias)")

**Step 2: the whole network, layer by layer** (torchvision's single-GPU version of AlexNet):

In [ ]:
spec = [("conv1", 64, 11, 4, 2), ("pool1", None, 3, 2, 0), ("conv2", 192, 5, 1, 2), ("pool2", None, 3, 2, 0),
        ("conv3", 384, 3, 1, 1), ("conv4", 256, 3, 1, 1), ("conv5", 256, 3, 1, 1), ("pool3", None, 3, 2, 0)]
h, c = 224, 3                                  # current spatial size and number of channels
total_params = total_macs = 0
print(f"{'layer':7s} {'output':>13s} {'params':>12s} {'mult-adds':>14s}")
for name, c_out, k, s, p in spec:
    h = conv_out(h, k, s, p)
    if c_out is None:                          # a pooling layer: no parameters, no multiplies
        print(f"{name:7s} {f'{c}x{h}x{h}':>13s} {0:>12,d} {0:>14,d}")
        continue
    params = c_out * (c * k * k + 1)
    macs = h * h * c_out * (c * k * k)
    total_params += params; total_macs += macs
    c = c_out
    print(f"{name:7s} {f'{c}x{h}x{h}':>13s} {params:>12,d} {macs:>14,d}")
fc_params = 0
for name, n_in, n_out in [("fc6", c * h * h, 4096), ("fc7", 4096, 4096), ("fc8", 4096, 1000)]:
    params, macs = n_out * (n_in + 1), n_in * n_out
    total_params += params; total_macs += macs; fc_params += params
    print(f"{name:7s} {n_out:>13,d} {params:>12,d} {macs:>14,d}")
print(f"{'total':7s} {'':>13s} {total_params:>12,d} {total_macs:>14,d}")
print(f"\n{total_params / 1e6:.1f}M parameters ({fc_params / total_params:.0%} in the fully connected layers), "
      f"{total_macs / 1e9:.2f} billion multiply-adds per image")

**What to notice:** **61.1 million parameters** (the published figure) and about **0.71 billion multiply-adds** per image. The **convolution layers do most of the computation with few parameters** (the same small filters are reused at every position), while the **fully connected layers hold most of the parameters with little computation**. That imbalance shaped later architectures (Lecture 8).

**Step 3 (🔎): how long does training take?** Training costs about **3× a forward pass** (forward + backward), and one multiply-add is **2 floating-point operations (FLOPs)**. AlexNet trained for about 90 passes ("epochs") over 1.2 million images. A GPU's **peak** speed is rarely reached; we assume 30% of it.

In [ ]:
flops_per_image = 3 * 2 * total_macs                          # forward + backward, 2 FLOPs per multiply-add
total_flops = flops_per_image * 1.2e6 * 90                     # 1.2M images x 90 epochs
print(f"total training compute: {total_flops:.2e} FLOPs")
for gpu, peak_flops_per_sec, n_gpus in [("2x GTX 580 (2012, FP32)", 1.58e12, 2),
                                         ("1x V100 (2017, FP16 tensor cores)", 125e12, 1),
                                         ("1x H100 (2022, BF16 tensor cores)", 989e12, 1)]:
    seconds = total_flops / (peak_flops_per_sec * n_gpus * 0.3)
    print(f"{gpu:36s}: ~{seconds / 86400:5.2f} days  ({seconds / 3600:6.1f} hours)")

**What to notice:** the estimate for 2012 hardware lands right at the reported **5–6 days**, a good check on the arithmetic. The same job on one modern GPU takes under an hour: about a **thousand-fold** drop in a decade.

---
## 6. 2012 to present: deep learning everywhere

**The explosion:** submissions to CVPR (the top computer-vision conference) grew from ~2,000 a year in the early 2010s to over 10,000 in the 2020s. The **2018 Turing Award** went to **Yoshua Bengio, Geoffrey Hinton, and Yann LeCun** for deep learning.

**Architectures:** AlexNet (2012) → ResNet (2015) → **Vision Transformer** (ViT, 2021) → **Diffusion Transformer** (DiT, 2023) **[CS231n 2026]**.

**Tasks deep learning now handles:**
* **Classification** and **retrieval** (find similar images)
* **Object detection** (Faster R-CNN, 2015) and **segmentation** (Segment Anything, 2023 **[CS231n 2026]**)
* **Video** classification and activity recognition; **pose** estimation; **3-D human mesh** reconstruction **[CS231n 2026]**
* **Scene graphs**: objects and how they relate (*person wears shirt, person on snow*)
* **Game playing** from pixels (Atari; AlphaGo/AlphaZero)
* **Science and medicine:** medical imaging, galaxy classification, protein structure (AlphaFold, 2021)
* **Captioning:** early systems (2015) wrote one-line captions (*"A white teddy bear sitting in the grass"*); a 2026 model (Gemini 3, in the CS231n slides) writes a paragraph about the same photo, noticing the bear's knitted sweater and a daisy on its head **[CS231n 2026]**
* **Generation:** GANs (2018), DALL·E (2021), and by 2026 photorealistic text-to-image and text-to-video models **[CS231n 2026]**

### 6.1 The three ingredients: computation, algorithms, data
Why 2012, when LeNet's recipe existed in 1998?
* **Computation:** GPUs got enormously faster and cheaper (§5.7: days → minutes), and they're ideal for the matrix math networks need (Lecture 9).
* **Algorithms:** backprop and CNNs, plus practical improvements (ReLU, dropout, good initialization, batch normalization, residual connections; Lectures 7–11).
* **Data:** large labeled datasets like ImageNet, made possible by the internet.

None was enough alone; together they produced the jump.

---
## 7. A long way to go: limits, harms, and benefits

* **Understanding is still shallow.** Andrej Karpathy's 2012 example: a photo of President Obama secretly pressing his foot on a scale while an aide weighs himself, as onlookers laugh. Getting the joke needs physics (feet add weight), intentions, and social knowledge, which pattern recognition alone doesn't give.
* **[CS231n 2026]:** there's a lot we don't know how to do; data is critical; visual intelligence means **understanding, reasoning, and generation**.

### 7.1 Computer vision can cause harm
* **Harmful stereotypes**, and systems that **affect people's lives**, e.g., AI job-interview systems that analyzed applicants' faces (example credit: Timnit Gebru).
* **Gender Shades** (Buolamwini & Gebru, 2018) tested commercial face-analysis systems: error rates reached **34.7% for darker-skinned women**, while the worst rate for **lighter-skinned men was 0.8%**.

**🔎 Always measure performance per group.** An overall accuracy can look fine while one group fares badly, simply because that group is a small part of the test set. The overall error is a **weighted average** of the group errors:
$$\text{overall error} = \sum_g(\text{share of group }g)\times(\text{error rate of group }g)$$

In [ ]:
groups = {                                  # (share of the test set, error rate); illustrative mix
    "lighter-skinned men":   (0.45, 0.008),
    "lighter-skinned women": (0.25, 0.07),
    "darker-skinned men":    (0.20, 0.06),
    "darker-skinned women":  (0.10, 0.347),
}
overall = sum(share * err for share, err in groups.values())
print(f"overall error rate: {overall:.1%}  (looks fine)")
for name, (share, err) in groups.items():
    print(f"  {name:22s}: {share:4.0%} of the test set, error {err:5.1%}")
print("(the largest and smallest error rates are those Gender Shades reported; the group shares are made up)")

### 7.2 Computer vision can save lives **[CS231n 2026]**
* **Healthcare:** contactless sensors with vision to monitor patients in intensive care, operating rooms, and senior care (detecting falls, checking hand hygiene).
* **Robotics:** robots doing household tasks.
* **Spatial intelligence:** a "virtuous cycle of seeing, learning, and doing"; e.g., **world models** that reconstruct or generate 3-D scenes from a few images.

---
## 8. Course overview

### 8.1 The UMich lectures and their CS231n 2026 counterparts
| UMich EECS 498 (Fall 2019) | CS231n 2026 | What CS231n 2026 adds |
|---|---|---|
| 1. Introduction | Lecture 1 | ViT/DiT, Segment Anything, spatial intelligence |
| 2. Image Classification | Lecture 2 | |
| 3. Linear Classifiers | Lecture 2 | |
| 4. Optimization | Lecture 3 | learning-rate schedules, AdamW |
| 5. Neural Networks | Lecture 4 | GELU |
| 6. Backpropagation | Lecture 4 | |
| 7. Convolutional Networks | Lecture 5 | |
| 8. CNN Architectures | Lecture 6 | |
| 9. Hardware and Software | Lecture 1 part 2 | distributed training |
| 10–11. Training Neural Networks | Lecture 6 | |
| 12. Recurrent Networks | Lecture 7 | |
| 13. Attention | Lecture 8 | Transformers, Vision Transformers |
| 14. Visualizing and Understanding | Lecture 9 | |
| 15–16. Detection and Segmentation | Lecture 9 | Segment Anything |
| 17. 3D Vision | Lectures 13–17 | 3-D generation, world models |
| 18. Videos | Lecture 10 | multimodal video |
| 19–20. Generative Models | Lectures 13–14 | **diffusion models** |
| 21. Reinforcement Learning | Lectures 15–17 | robot learning |
| 22. Conclusion | Lecture 18 | human-centered AI |

(CS231n numbers are approximate.) Some 2026 topics postdate the 2019 lectures (notably **self-supervised learning**, **vision–language models** like CLIP, and **diffusion**); this series adds them where they fit.

### 8.2 What CS231n 2026 emphasizes **[CS231n 2026]**
* **Deep learning basics:** image classification, linear classifiers, regularization and optimization, neural networks.
* **Perceiving the visual world:** tasks beyond classification:
  * **semantic segmentation**: label every pixel (*grass, cat, tree, sky*);
  * **object detection**: a box around each object (*dog, dog, cat*);
  * **instance segmentation**: the exact pixels of each object.
  * Models beyond the MLP: CNNs, RNNs, Transformers. **Distributed training:** *data parallelism* (each GPU holds a full copy of the model and processes different data) vs. *model parallelism* (the model itself is split across GPUs).
* **Generative and interactive vision:** self-supervised learning (no human labels), generative models (diffusion, text-to-image/video), vision–language models (CLIP), 3-D vision, embodied AI (robots).
* **Human-centered applications and implications.**
* **Goals:** turn vision problems into well-defined tasks; build, debug, and train models in PyTorch; understand recent research, open problems, and societal impact.

### 8.3 Prerequisites and resources
Comfortable **Python** and **NumPy** (later PyTorch); **calculus** and **linear algebra**. Optional books (from CS231n 2026): *Deep Learning* (Goodfellow, Bengio & Courville), *Mathematics for Machine Learning*, *Dive into Deep Learning*. 🔎 For the classical vision in §3: Szeliski, *Computer Vision: Algorithms and Applications* (free online).

---
## 9. 🔎 Practical setup: images as tensors

You'll constantly move images between libraries, and the conventions differ:
* **Image files, NumPy, PIL, OpenCV:** **height × width × channels (HWC)**, usually **uint8** in [0, 255].
* **OpenCV** stores channels as **BGR**, not RGB.
* **PyTorch models:** **channels × height × width (CHW)**, batched as **N × C × H × W**, as **float32**.
* **Pretrained ImageNet models** expect pixels scaled to [0, 1], then **normalized per channel** with mean $(0.485, 0.456, 0.406)$ and standard deviation $(0.229, 0.224, 0.225)$.

Converting step by step. **Step 1: an image as it comes from a file.**

In [ ]:
file_rng = np.random.default_rng(0)
img_hwc = (file_rng.random((224, 224, 3)) * 255).astype(np.uint8)
print("from file:", img_hwc.shape, img_hwc.dtype, " range", img_hwc.min(), "-", img_hwc.max())

**Step 2: to floats in [0, 1], then normalize each channel.** NumPy **broadcasting** lets the 3-element mean and std apply to every pixel: an array of shape (224, 224, 3) minus an array of shape (3,) subtracts along the last axis.

In [ ]:
mean = np.array([0.485, 0.456, 0.406]); std = np.array([0.229, 0.224, 0.225])
x = img_hwc.astype(np.float32) / 255.0          # uint8 0..255 -> float 0..1
x = (x - mean) / std                            # per-channel normalization (broadcast over H and W)
print("after normalization: mean per channel", x.mean(axis=(0, 1)).round(2))

**Step 3: reorder to CHW and add a batch dimension.** `transpose(2, 0, 1)` moves axis 2 (channels) to the front; `x[None]` adds a new leading axis of size 1.

In [ ]:
x_chw = x.transpose(2, 0, 1)                    # (H, W, C) -> (C, H, W)
batch = x_chw[None].astype(np.float32)          # (C, H, W) -> (1, C, H, W)
print("model input:", batch.shape, batch.dtype)
bgr = img_hwc[:, :, ::-1]                       # RGB <-> BGR is just reversing the channel axis
print("BGR -> RGB recovers the original:", np.array_equal(bgr[:, :, ::-1], img_hwc))

**The same in PyTorch** (run in Colab), plus a check that torchvision's AlexNet has the parameter count we computed in §5.7:

In [ ]:
# [PyTorch] Needs: pip install torch torchvision
import torch, torchvision
t = torch.from_numpy(img_hwc).permute(2, 0, 1).float() / 255      # HWC uint8 -> CHW float in [0, 1]
normalize = torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
batch_t = normalize(t).unsqueeze(0)                                # add the batch dimension
print(batch_t.shape, batch_t.dtype)
alexnet = torchvision.models.alexnet(weights=None)                 # architecture only, no download
print("torchvision AlexNet parameters:", sum(p.numel() for p in alexnet.parameters()))   # compare with Sec. 5.7
torch.manual_seed(0)                                               # for reproducible results

---
## 10. Summary

| Era | Key idea | What we built | Why it wasn't enough |
|---|---|---|---|
| 1959 Hubel & Wiesel | Simple cells (oriented, one place) → complex cells (pooled) | Gabor simple cells; max-pooled complex cells | (It inspired CNNs) |
| 1963 Roberts | Edges → 3-D blocks | Roberts cross | Fails with noise and clutter |
| 1970s Marr | Primal sketch → 2½-D sketch → 3-D model | Normals and shading from depth | A framework, not a recipe |
| 1970s parts | Parts + springs | 3-part pictorial structure | Hand-designed models |
| 1986 Canny | Smooth → gradient → thin → link | Full Canny detector | Edges ≠ objects |
| 1997 Normalized cuts | Group similar nearby pixels | Affinities + eigenvector | Regions, not categories |
| 1999 SIFT | Match rotation/lighting-invariant patches | Harris + orientation histograms + ratio test | Hand-designed features |
| 2001 Viola–Jones | Boosted rectangle features, cascade | Integral image + AdaBoost | One category; hand-designed features |
| 1958–69 Perceptron | Learn a linear classifier | Mark-I-style perceptron | No XOR, no translation invariance |
| 1980–98 | Neocognitron → backprop → LeNet | XOR with a hidden layer | Little data, slow compute |
| 2012 AlexNet | Deep CNN + GPUs + ImageNet | Parameter/compute/training-time accounting | (The start of this course) |

### Check yourself
<details><summary>1. What do simple and complex cells correspond to in a CNN?</summary>

Simple cells ≈ convolution filters (oriented detectors at one location). Complex cells ≈ pooling (combining detector outputs over nearby locations), which gives translation invariance.
</details>

<details><summary>2. Why does the Roberts operator fail on noisy images, and which Canny steps fix it?</summary>

It takes differences of raw neighboring pixels, and differences amplify noise. Canny smooths first, thins responses with non-maximum suppression, and keeps only weak edges connected to strong ones (hysteresis).
</details>

<details><summary>3. In normalized cuts, why normalize the cut by each group's total connections?</summary>

A plain minimum cut favors slicing off a few isolated pixels (few edges to cut). Dividing by each side's total connection strength makes tiny groups expensive.
</details>

<details><summary>4. What does Lowe's ratio test reject, and why?</summary>

Matches whose best candidate isn't clearly better than the second best (distance ratio ≥ 0.8). Those patches are ambiguous, so the match is likely wrong.
</details>

<details><summary>5. Why can't a perceptron separate vertical from horizontal lines placed anywhere?</summary>

Summing the "vertical lines score positive" conditions over all columns, and the "horizontal lines score negative" conditions over all rows, puts the same total weight both above and below −20b: a contradiction.
</details>

<details><summary>6. In AlexNet, where are the parameters, and where is the computation?</summary>

~96% of the 61M parameters are in the fully connected layers; most of the ~0.7 billion multiply-adds are in the convolution layers.
</details>

### Exercises
1. Add a second simple cell (preferring 90°) to §3.1, and use the two complex-cell responses to classify a bar's orientation at any position.
2. Plot Canny's and Roberts' F1 against noise level (0.02 to 0.2). Where does each break down?
3. Add a third region to §3.8's image and apply normalized cuts recursively (split each segment again).
4. In §3.9, change the rotation to 60° or 90° and the brightness factor to 2. Which part of the descriptor provides each kind of invariance?
5. Add three-rectangle features ("dark–light–dark") to §3.10. Does AdaBoost need fewer rounds?
6. Using §5.7's formulas, compute VGG-16's parameters and multiply-adds (13 3×3 conv layers with channels 64, 64, 128, 128, 256×3, 512×3, 512×3, pooling between blocks, then FC 25088→4096→4096→1000). Where are its parameters?

### References
* Hubel & Wiesel 1959; Daugman 1985 (Gabor models); Roberts 1963; Papert 1966 (MIT AI Memo 100); Marr 1982, *Vision*.
* Brooks & Binford 1979; Fischler & Elschlager 1973; Felzenszwalb et al. 2010 (deformable part models).
* Canny 1986; Lowe 1987; Shi & Malik 1997/2000; Harris & Stephens 1988; Lowe 1999/2004 (SIFT); Viola & Jones 2001; Freund & Schapire 1997 (AdaBoost).
* Biederman 1972; Potter 1976; Thorpe et al. 1996; Kanwisher et al. 1997; Epstein & Kanwisher 1998.
* Fei-Fei et al. 2004 (Caltech101); Everingham et al. 2010 (PASCAL); Deng et al. 2009 (ImageNet); Russakovsky et al. 2015 (ILSVRC).
* Rosenblatt 1958; Minsky & Papert 1969; Fukushima 1980; Rumelhart, Hinton & Williams 1986; LeCun et al. 1998; Hinton & Salakhutdinov 2006; Glorot & Bengio 2010; Krizhevsky et al. 2012.
* Buolamwini & Gebru 2018 (*Gender Shades*); Szeliski 2022, *Computer Vision: Algorithms and Applications*.

**Next:** Lecture 2, Image Classification.